In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Project directory
PROJECT_DIR = Path(
    r"D:\SVKM NMIMS Indore\Academic Related\Research Papers"
    r"\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised"
)

# Data directories
INTEGRATED_DIR = (
    PROJECT_DIR / "data" / "intermediate" / "integrated_climate"
)

SPATIAL_DIR = PROJECT_DIR / "data" / "spatial"
RESULTS_DIR = PROJECT_DIR / "results" / "model_ready_data"

# Create output directories
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Identify integrated annual files
integrated_files = sorted(
    INTEGRATED_DIR.glob("integrated_climate_*.nc")
)

print("Project directory:", PROJECT_DIR)
print("Integrated data directory:", INTEGRATED_DIR)
print("Number of annual files:", len(integrated_files))
print("Results directory:", RESULTS_DIR)

assert len(integrated_files) == 26, (
    "Expected 26 annual integrated climate files."
)

print("\nNotebook setup completed successfully.")

Project directory: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised
Integrated data directory: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\intermediate\integrated_climate
Number of annual files: 26
Results directory: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\results\model_ready_data

Notebook setup completed successfully.


In [2]:
# Step 2: Load integrated climate data

annual_datasets = [
    xr.open_dataset(file)
    for file in integrated_files
]

climate_ds = xr.concat(
    annual_datasets,
    dim="time"
).sortby("time")

# Extract climate variables
dates = pd.to_datetime(climate_ds.time.values)

rainfall_data = climate_ds["rainfall"].values
temperature_data = climate_ds["temperature"].values

# Joint validity mask
joint_mask = (
    np.isfinite(rainfall_data)
    & np.isfinite(temperature_data)
)

# Display dataset information
print("Integrated dataset loaded successfully.")
print("\nDataset dimensions:")
print(climate_ds.sizes)

print("\nVariables:")
print(list(climate_ds.data_vars))

print("\nTemporal coverage:")
print("Start date:", dates.min())
print("End date:", dates.max())
print("Total days:", len(dates))

print("\nData array shapes:")
print("Rainfall:", rainfall_data.shape)
print("Temperature:", temperature_data.shape)
print("Joint validity mask:", joint_mask.shape)

print("\nJoint valid observations:", joint_mask.sum())
print("Joint missing observations:", (~joint_mask).sum())

assert rainfall_data.shape == (9497, 5040)
assert temperature_data.shape == (9497, 5040)
assert len(dates) == 9497

Integrated dataset loaded successfully.

Dataset dimensions:
Frozen({'time': 9497, 'node': 5040})

Variables:
['rainfall', 'temperature']

Temporal coverage:
Start date: 2000-01-01 00:00:00
End date: 2025-12-31 00:00:00
Total days: 9497

Data array shapes:
Rainfall: (9497, 5040)
Temperature: (9497, 5040)
Joint validity mask: (9497, 5040)

Joint valid observations: 41284549
Joint missing observations: 6580331


In [3]:
# Step 3: Missing-data summary

total_observations = joint_mask.size
valid_observations = joint_mask.sum()
missing_observations = total_observations - valid_observations

# Node-wise missingness
node_valid_days = joint_mask.sum(axis=0)
node_missing_days = joint_mask.shape[0] - node_valid_days

# Daily missingness
daily_valid_nodes = joint_mask.sum(axis=1)
daily_missing_nodes = joint_mask.shape[1] - daily_valid_nodes

print("OVERALL JOINT AVAILABILITY")
print(f"Valid observations: {valid_observations:,}")
print(f"Missing observations: {missing_observations:,}")
print(f"Availability: {100 * valid_observations / total_observations:.2f}%")

print("\nNODE-WISE AVAILABILITY")
print("Nodes with at least one valid observation:",
      np.sum(node_valid_days > 0))
print("Nodes with zero valid observations:",
      np.sum(node_valid_days == 0))
print("Nodes with 100% valid observations:",
      np.sum(node_valid_days == len(dates)))

print("\nDAILY AVAILABILITY")
print(f"Mean valid nodes: {daily_valid_nodes.mean():.2f}")
print(f"Minimum valid nodes: {daily_valid_nodes.min()}")
print(f"Maximum valid nodes: {daily_valid_nodes.max()}")

print("\nMISSINGNESS PERCENTILES (NODE-WISE)")
print(np.percentile(
    100 * node_missing_days / len(dates),
    [0, 25, 50, 75, 90, 100]
))

OVERALL JOINT AVAILABILITY
Valid observations: 41,284,549
Missing observations: 6,580,331
Availability: 86.25%

NODE-WISE AVAILABILITY
Nodes with at least one valid observation: 4365
Nodes with zero valid observations: 675
Nodes with 100% valid observations: 3747

DAILY AVAILABILITY
Mean valid nodes: 4347.11
Minimum valid nodes: 4046
Maximum valid nodes: 4365

MISSINGNESS PERCENTILES (NODE-WISE)
[0.00000000e+00 0.00000000e+00 0.00000000e+00 1.05296409e-02
 1.00000000e+02 1.00000000e+02]


In [4]:
# Step 4: Variable-wise missing-data analysis

rainfall_mask = np.isfinite(rainfall_data)
temperature_mask = np.isfinite(temperature_data)

variable_missing_summary = pd.DataFrame({
    "variable": ["Rainfall", "Temperature"],
    "valid_observations": [
        rainfall_mask.sum(),
        temperature_mask.sum()
    ],
    "missing_observations": [
        (~rainfall_mask).sum(),
        (~temperature_mask).sum()
    ],
    "availability_percentage": [
        100 * rainfall_mask.mean(),
        100 * temperature_mask.mean()
    ],
    "nodes_with_valid_data": [
        np.sum(rainfall_mask.sum(axis=0) > 0),
        np.sum(temperature_mask.sum(axis=0) > 0)
    ],
    "nodes_with_no_valid_data": [
        np.sum(rainfall_mask.sum(axis=0) == 0),
        np.sum(temperature_mask.sum(axis=0) == 0)
    ]
})

display(variable_missing_summary)

# Verify that the masks are consistent with the integrated dataset
assert np.array_equal(
    rainfall_mask & temperature_mask,
    joint_mask
)

print("Variable-wise missingness verification passed.")

,variable,valid_observations,missing_observations,availability_percentage,nodes_with_valid_data,nodes_with_no_valid_data
0,Rainfall,44977724,2887156,93.968112,4736,304
1,Temperature,42324403,5540477,88.424755,4476,564


Variable-wise missingness verification passed.


In [5]:
# Step 5: Define chronological train-validation-test split

train_mask = (dates.year >= 2000) & (dates.year <= 2017)

validation_mask = (dates.year >= 2018) & (dates.year <= 2023)

test_mask = (dates.year >= 2024) & (dates.year <= 2025)

# Extract dates for each split
train_dates = dates[train_mask]
validation_dates = dates[validation_mask]
test_dates = dates[test_mask]

# Create split summary
temporal_split_summary = pd.DataFrame({
    "dataset": ["Training", "Validation", "Testing"],
    "start_date": [
        train_dates.min(),
        validation_dates.min(),
        test_dates.min()
    ],
    "end_date": [
        train_dates.max(),
        validation_dates.max(),
        test_dates.max()
    ],
    "number_of_days": [
        len(train_dates),
        len(validation_dates),
        len(test_dates)
    ]
})

display(temporal_split_summary)

# Verify chronological separation and complete coverage
assert train_mask.sum() + validation_mask.sum() + test_mask.sum() == len(dates)

assert not np.any(train_mask & validation_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(validation_mask & test_mask)

assert train_dates.max() < validation_dates.min()
assert validation_dates.max() < test_dates.min()

print("\nTemporal split verification passed.")
print("Total days:", len(dates))
print("Training days:", len(train_dates))
print("Validation days:", len(validation_dates))
print("Testing days:", len(test_dates))

,dataset,start_date,end_date,number_of_days
0,Training,2000-01-01,2017-12-31,6575
1,Validation,2018-01-01,2023-12-31,2191
2,Testing,2024-01-01,2025-12-31,731



Temporal split verification passed.
Total days: 9497
Training days: 6575
Validation days: 2191
Testing days: 731


In [6]:
# Step 6: Missing-data summary for each temporal split

split_masks = {
    "Training": train_mask,
    "Validation": validation_mask,
    "Testing": test_mask
}

split_missing_summary = []

for split_name, split_mask in split_masks.items():

    rainfall_split = rainfall_data[split_mask]
    temperature_split = temperature_data[split_mask]

    rainfall_valid = np.isfinite(rainfall_split)
    temperature_valid = np.isfinite(temperature_split)

    joint_valid = rainfall_valid & temperature_valid

    total = joint_valid.size

    split_missing_summary.append({
        "dataset": split_name,
        "days": split_mask.sum(),
        "rainfall_availability_pct":
            100 * rainfall_valid.sum() / rainfall_valid.size,
        "temperature_availability_pct":
            100 * temperature_valid.sum() / temperature_valid.size,
        "joint_availability_pct":
            100 * joint_valid.sum() / total,
        "joint_valid_observations": joint_valid.sum(),
        "joint_missing_observations": total - joint_valid.sum()
    })

split_missing_summary = pd.DataFrame(split_missing_summary)

display(
    split_missing_summary.round(3)
)

,dataset,days,rainfall_availability_pct,temperature_availability_pct,joint_availability_pct,joint_valid_observations,joint_missing_observations
0,Training,6575,93.968,88.322,86.159,28551469,4586531
1,Validation,2191,93.968,88.623,86.431,9544281,1498359
2,Testing,731,93.968,88.754,86.552,3188799,495441


In [7]:
# Step 7: Training-period node-wise availability

rainfall_train_mask = rainfall_mask[train_mask]
temperature_train_mask = temperature_mask[train_mask]

node_availability_summary = pd.DataFrame({
    "node_index": np.arange(5040),
    "rainfall_valid_days": rainfall_train_mask.sum(axis=0),
    "temperature_valid_days": temperature_train_mask.sum(axis=0)
})

# Classify availability
for variable in ["rainfall", "temperature"]:

    valid_days = node_availability_summary[
        f"{variable}_valid_days"
    ]

    node_availability_summary[
        f"{variable}_category"
    ] = np.select(
        [
            valid_days == 0,
            valid_days == len(train_dates)
        ],
        [
            "No observations",
            "Complete observations"
        ],
        default="Partial observations"
    )

# Summarise categories
for variable in ["rainfall", "temperature"]:

    print(f"\n{variable.upper()} NODE AVAILABILITY")

    print(
        node_availability_summary[
            f"{variable}_category"
        ].value_counts()
    )

# Save detailed results
node_availability_summary.to_csv(
    RESULTS_DIR / "training_node_availability.csv",
    index=False
)

print("\nTraining node availability saved.")


RAINFALL NODE AVAILABILITY
rainfall_category
Complete observations    4736
No observations           304
Name: count, dtype: int64

TEMPERATURE NODE AVAILABILITY
temperature_category
Complete observations    3889
Partial observations      587
No observations           564
Name: count, dtype: int64

Training node availability saved.


In [8]:
# Step 8: Examine missing-value patterns in the training period

rainfall_train = rainfall_data[train_mask]
temperature_train = temperature_data[train_mask]

# Count missing observations
rainfall_missing_per_day = (
    ~np.isfinite(rainfall_train)
).sum(axis=0)

temperature_missing_per_day = (
    ~np.isfinite(temperature_train)
).sum(axis=0)

# Examine partial temperature nodes only
partial_temperature_nodes = node_availability_summary.loc[
    node_availability_summary["temperature_category"]
    == "Partial observations",
    "node_index"
].values

# Missing-value summary for partially observed temperature nodes
partial_temperature_missing = (
    ~np.isfinite(
        temperature_train[:, partial_temperature_nodes]
    )
)

missing_days_per_node = partial_temperature_missing.sum(axis=0)

print("PARTIALLY OBSERVED TEMPERATURE NODES")
print("Number of nodes:", len(partial_temperature_nodes))
print("Mean missing days per node:", missing_days_per_node.mean())
print("Minimum missing days:", missing_days_per_node.min())
print("Maximum missing days:", missing_days_per_node.max())

print("\nMISSING OBSERVATIONS ACROSS ALL NODES")
print("Rainfall:", (~np.isfinite(rainfall_train)).sum())
print("Temperature:", (~np.isfinite(temperature_train)).sum())

print("\nTemperature missingness percentage in partial nodes:",
      100 * partial_temperature_missing.mean())

PARTIALLY OBSERVED TEMPERATURE NODES
Number of nodes: 587
Mean missing days per node: 275.1243611584327
Minimum missing days: 1
Maximum missing days: 3273

MISSING OBSERVATIONS ACROSS ALL NODES
Rainfall: 1998800
Temperature: 3869798

Temperature missingness percentage in partial nodes: 4.1844009301662775


In [9]:
# Step 9: Analyse consecutive missing temperature gaps

def calculate_missing_gap_lengths(mask):
    """
    Calculate lengths of consecutive missing periods.
    mask: Boolean array where True means valid observation.
    """
    missing = ~mask

    padded = np.concatenate(([False], missing, [False]))
    changes = np.diff(padded.astype(int))

    starts = np.where(changes == 1)[0]
    ends = np.where(changes == -1)[0]

    return ends - starts


gap_lengths = []

for node in partial_temperature_nodes:

    node_mask = np.isfinite(
        temperature_train[:, node]
    )

    gaps = calculate_missing_gap_lengths(node_mask)

    gap_lengths.extend(gaps)

gap_lengths = np.array(gap_lengths)

gap_summary = pd.DataFrame({
    "gap_category": [
        "1 day",
        "2–3 days",
        "4–7 days",
        "8–30 days",
        "More than 30 days"
    ],
    "number_of_gaps": [
        np.sum(gap_lengths == 1),
        np.sum((gap_lengths >= 2) & (gap_lengths <= 3)),
        np.sum((gap_lengths >= 4) & (gap_lengths <= 7)),
        np.sum((gap_lengths >= 8) & (gap_lengths <= 30)),
        np.sum(gap_lengths > 30)
    ]
})

display(gap_summary)

print("\nTotal missing gaps:", len(gap_lengths))
print("Mean gap length:", gap_lengths.mean())
print("Median gap length:", np.median(gap_lengths))
print("Maximum gap length:", gap_lengths.max())

,gap_category,number_of_gaps
0,1 day,18217
1,2–3 days,2446
2,4–7 days,523
3,8–30 days,411
4,More than 30 days,344



Total missing gaps: 21941
Mean gap length: 7.36055785971469
Median gap length: 1.0
Maximum gap length: 2192


In [10]:
# Step 10: Identify nodes requiring special input treatment

rainfall_zero_nodes = np.where(
    rainfall_train_mask.sum(axis=0) == 0
)[0]

temperature_zero_nodes = np.where(
    temperature_train_mask.sum(axis=0) == 0
)[0]

# Nodes with no observations in either variable
both_zero_nodes = np.intersect1d(
    rainfall_zero_nodes,
    temperature_zero_nodes
)

# Nodes with no rainfall but some temperature
rainfall_zero_only = np.setdiff1d(
    rainfall_zero_nodes,
    both_zero_nodes
)

# Nodes with no temperature but some rainfall
temperature_zero_only = np.setdiff1d(
    temperature_zero_nodes,
    both_zero_nodes
)

print("NODES WITH NO TRAINING OBSERVATIONS")
print("Rainfall:", len(rainfall_zero_nodes))
print("Temperature:", len(temperature_zero_nodes))
print("Both variables:", len(both_zero_nodes))
print("Rainfall unavailable only:", len(rainfall_zero_only))
print("Temperature unavailable only:", len(temperature_zero_only))

print("\nVerification:")
print(
    "Classification complete:",
    len(both_zero_nodes)
    + len(rainfall_zero_only)
    + len(temperature_zero_only)
    == len(np.union1d(rainfall_zero_nodes, temperature_zero_nodes))
)

NODES WITH NO TRAINING OBSERVATIONS
Rainfall: 304
Temperature: 564
Both variables: 193
Rainfall unavailable only: 111
Temperature unavailable only: 371

Verification:
Classification complete: True


In [11]:
# Step 11: Load and inspect original spatial node information

nodes_file = SPATIAL_DIR / "grid_nodes.csv"

nodes_df = pd.read_csv(nodes_file)

print("Spatial node file:", nodes_file)
print("\nShape:", nodes_df.shape)
print("\nColumns:")
print(nodes_df.columns.tolist())

print("\nFirst five records:")
display(nodes_df.head())

print("\nCoordinate summary:")
display(
    nodes_df[["latitude", "longitude"]].describe()
)

# Verify original node count
assert len(nodes_df) == 5040

print("\nOriginal 5,040 spatial nodes verified.")

Spatial node file: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\spatial\grid_nodes.csv

Shape: (5040, 2)

Columns:
['latitude', 'longitude']

First five records:


,latitude,longitude
0,8.451258,76.834814
1,8.733465,76.608342
2,8.631885,76.770510
3,8.885523,76.547012
4,8.858651,76.733590



Coordinate summary:


,latitude,longitude
count,5040.000000,5040.000000
mean,23.364367,79.686979
std,6.119055,6.131353
min,6.880740,68.465904
25%,19.608651,75.483590
50%,23.858651,78.233590
75%,27.108651,82.733590
max,37.002270,97.398749



Original 5,040 spatial nodes verified.


In [12]:
# Step 12: Locate existing graph files

graph_files = []

for extension in ["*.pt", "*.npy", "*.npz", "*.csv", "*.pkl"]:
    graph_files.extend(SPATIAL_DIR.glob(extension))

print("Graph-related files found:")

for file in graph_files:
    print(
        f"{file.name} | "
        f"{file.stat().st_size / (1024 ** 2):.2f} MB"
    )

print("\nSpatial directory contents:")
for file in sorted(SPATIAL_DIR.iterdir()):
    print(file.name)

Graph-related files found:
static_spatial_graph_verified.pt | 0.62 MB
grid_nodes.csv | 0.12 MB
rainfall_bilinear_mapping.csv | 0.44 MB
rainfall_interpolation_coverage.csv | 0.15 MB
rainfall_node_coverage.csv | 0.26 MB
temperature_bilinear_mapping.csv | 0.41 MB

Spatial directory contents:
centroids.shp
grid.shp
grid_nodes.csv
rainfall_bilinear_mapping.csv
rainfall_interpolation_coverage.csv
rainfall_node_coverage.csv
static_spatial_graph_verified.pt
temperature_bilinear_mapping.csv


In [13]:
# Step 13: Search the project for original graph artefacts

graph_keywords = [
    "edge_index",
    "adjacency",
    "graph",
    "knn",
    "spatial_graph",
    "data.pt",
    "graph_data"
]

matching_files = []

for file in PROJECT_DIR.rglob("*"):
    if not file.is_file():
        continue

    if file.suffix.lower() in [
        ".pt", ".pth", ".npy", ".npz",
        ".pkl", ".pickle", ".csv"
    ]:
        if any(
            keyword in file.name.lower()
            for keyword in graph_keywords
        ):
            matching_files.append(file)

print("Potential graph-related files:\n")

for file in matching_files:
    print(file.relative_to(PROJECT_DIR))

print("\nTotal matching files:", len(matching_files))

Potential graph-related files:

data\spatial\static_spatial_graph_verified.pt

Total matching files: 1


In [14]:
# Step 15: Inspect the original spatial graph

import torch

graph_file = r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final Codes\static_spatial_graph.pt"

graph = torch.load(
    graph_file,
    map_location="cpu",
    weights_only=False
)

print("Graph type:", type(graph))

if hasattr(graph, "edge_index"):
    print("Number of nodes:", graph.num_nodes)
    print("Edge index shape:", graph.edge_index.shape)
    print("Number of edges:", graph.edge_index.shape[1])
    print("Node features:", graph.x.shape if graph.x is not None else None)
    print("Edge attributes:", graph.edge_attr.shape if graph.edge_attr is not None else None)

    print("\nFirst 10 edges:")
    print(graph.edge_index[:, :10])
else:
    print(graph)

C:\Users\rajgm\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Graph type: <class 'torch_geometric.data.data.Data'>
Number of nodes: 5040
Edge index shape: torch.Size([2, 40320])
Number of edges: 40320
Node features: None
Edge attributes: None

First 10 edges:
tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   1,   1],
        [  2,   4, 136, 126, 138,   5, 141,  10,   3,   4]])


C:\Users\rajgm\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\data\data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


In [15]:
import torch
import numpy as np

edge_index = graph.edge_index.cpu()

num_nodes = graph.num_nodes
num_edges = edge_index.shape[1]

source = edge_index[0].numpy()
target = edge_index[1].numpy()

# 1. Check node index range
print("Minimum node index:", edge_index.min().item())
print("Maximum node index:", edge_index.max().item())

# 2. Check outgoing degree
out_degree = np.bincount(source, minlength=num_nodes)

print("\nOutgoing degree:")
print("Minimum:", out_degree.min())
print("Maximum:", out_degree.max())
print("Mean:", out_degree.mean())
print("Nodes with exactly 8 outgoing edges:",
      np.sum(out_degree == 8))

# 3. Check self-loops
self_loops = np.sum(source == target)
print("\nSelf-loops:", self_loops)

# 4. Check duplicate edges
edge_pairs = np.column_stack((source, target))
unique_edges = np.unique(edge_pairs, axis=0)

print("Unique edges:", len(unique_edges))
print("Duplicate edges:", num_edges - len(unique_edges))

# 5. Check reciprocal connections
edge_set = set(map(tuple, edge_pairs))
reciprocal_edges = sum(
    (v, u) in edge_set for u, v in edge_set
)

print("\nReciprocal directed edges:", reciprocal_edges)
print("Reciprocal percentage:",
      100 * reciprocal_edges / num_edges)

# 6. Basic validation
assert edge_index.min() >= 0
assert edge_index.max() < num_nodes
assert np.all(out_degree == 8)
assert len(unique_edges) == num_edges

print("\nBasic graph validation completed.")

Minimum node index: 0
Maximum node index: 5039

Outgoing degree:
Minimum: 8
Maximum: 8
Mean: 8.0
Nodes with exactly 8 outgoing edges: 5040

Self-loops: 0
Unique edges: 40320
Duplicate edges: 0

Reciprocal directed edges: 38080
Reciprocal percentage: 94.44444444444444

Basic graph validation completed.


In [16]:
import numpy as np
from sklearn.neighbors import BallTree

# Coordinates in radians: [latitude, longitude]
coordinates_rad = np.radians(
    nodes_df[["latitude", "longitude"]].to_numpy()
)

# Build an independent Haversine nearest-neighbour reference
tree = BallTree(coordinates_rad, metric="haversine")

# Query 9 neighbours: the node itself + its 8 nearest neighbours
distances, indices = tree.query(coordinates_rad, k=9)

# Exclude the node itself
expected_neighbours = [set(row[1:]) for row in indices]

# Extract neighbours from the recovered graph
actual_neighbours = [
    set(edge_index[1, edge_index[0] == node].cpu().numpy())
    for node in range(graph.num_nodes)
]

# Compare the two neighbour sets
matching_nodes = sum(
    actual_neighbours[i] == expected_neighbours[i]
    for i in range(graph.num_nodes)
)

partial_matches = [
    len(actual_neighbours[i] & expected_neighbours[i])
    for i in range(graph.num_nodes)
]

# Calculate geographic edge distances in kilometres
edge_distances_km = (
    distances[:, 1:] * 6371.0
)

print("Geographic k-NN verification")
print("--------------------------------")
print("Nodes with exact 8-neighbour match:",
      matching_nodes, "/", graph.num_nodes)
print("Exact match percentage:",
      round(100 * matching_nodes / graph.num_nodes, 2), "%")
print("Mean matching neighbours per node:",
      round(np.mean(partial_matches), 3), "/ 8")

print("\nGeographic edge distance (km):")
print("Minimum:", round(edge_distances_km.min(), 3))
print("Mean:", round(edge_distances_km.mean(), 3))
print("Maximum:", round(edge_distances_km.max(), 3))

C:\Users\rajgm\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\data\data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


Geographic k-NN verification
--------------------------------
Nodes with exact 8-neighbour match: 7 / 5040
Exact match percentage: 0.14 %
Mean matching neighbours per node: 2.768 / 8

Geographic edge distance (km):
Minimum: 3.858
Mean: 32.667
Maximum: 465.135


In [17]:
import torch
import numpy as np
from pathlib import Path
from sklearn.neighbors import BallTree

graph_files = [
    Path(r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final Codes\static_spatial_graph.pt"),
    Path(r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final Codes\Output from Notebook 2\final_gatv2_model.pt")
]

coordinates_rad = np.radians(
    nodes_df[["latitude", "longitude"]].to_numpy()
)

tree = BallTree(coordinates_rad, metric="haversine")
_, nearest_indices = tree.query(coordinates_rad, k=9)

expected = [set(row[1:]) for row in nearest_indices]

loaded_graphs = {}

for file in graph_files:
    print("\nFile:", file.name)

    if not file.exists():
        print("File not found")
        continue

    g = torch.load(file, map_location="cpu", weights_only=False)

    if not hasattr(g, "edge_index"):
        print("Not a graph Data object.")
        continue

    edges = g.edge_index.numpy()
    actual = [
        set(edges[1, edges[0] == i])
        for i in range(g.num_nodes)
    ]

    matches = sum(
        actual[i] == expected[i]
        for i in range(g.num_nodes)
    )

    mean_matches = np.mean([
        len(actual[i] & expected[i])
        for i in range(g.num_nodes)
    ])

    loaded_graphs[file.name] = edges

    print("Nodes:", g.num_nodes)
    print("Edges:", edges.shape[1])
    print("Exact geographic matches:", matches)
    print("Exact match percentage:",
          round(100 * matches / g.num_nodes, 2))
    print("Mean matching neighbours:",
          round(mean_matches, 3), "/ 8")

if len(loaded_graphs) == 2:
    names = list(loaded_graphs)
    same = np.array_equal(
        loaded_graphs[names[0]],
        loaded_graphs[names[1]]
    )

    print("\nAre both edge_index arrays identical?", same)


File: static_spatial_graph.pt


C:\Users\rajgm\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\data\data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


Nodes: 5040
Edges: 40320
Exact geographic matches: 7
Exact match percentage: 0.14
Mean matching neighbours: 2.768 / 8

File: final_gatv2_model.pt
Not a graph Data object.


In [18]:
from pathlib import Path
import pandas as pd

# Enter the original file path here
ORIGINAL_RAINFALL_FILE = Path(
    r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final-Year-Project\Notebooks\grid_with_rainfall.csv"
)

# Load the original file
rain_df_original = pd.read_csv(ORIGINAL_RAINFALL_FILE)

print("File loaded successfully")
print("Shape:", rain_df_original.shape)
print("Columns:", rain_df_original.columns.tolist())

# Display first five rows
display(rain_df_original.head())

# Check centroid availability
print("\nMissing centroid values:",
      rain_df_original["centroid"].isna().sum())

File loaded successfully
Shape: (5040, 2)
Columns: ['centroid', 'rainfall']


,centroid,rainfall
0,POINT (76.83481389058699 8.451258352481261),0.0
1,POINT (76.60834197748865 8.733465171526532),0.0
2,POINT (76.77051028642661 8.631884688242543),0.0
3,POINT (76.54701195556088 8.885522639737793),0.0
4,POINT (76.73359 8.858651),0.0



Missing centroid values: 0


In [19]:
import re
import numpy as np
from scipy.spatial.distance import cdist

# Extract latitude and longitude from original centroid strings
def parse_centroid(centroid):
    values = re.findall(
        r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",
        centroid
    )
    lon, lat = map(float, values)
    return lat, lon

original_coords = np.array([
    parse_centroid(value)
    for value in rain_df_original["centroid"]
])

current_coords = nodes_df[["latitude", "longitude"]].to_numpy()

# Compare coordinates row by row
rowwise_difference = np.linalg.norm(
    original_coords - current_coords, axis=1
)

print("Original coordinates:", original_coords.shape)
print("Current coordinates:", current_coords.shape)

print("\nRow-wise comparison:")
print("Exact matches:", np.sum(rowwise_difference == 0))
print("Matches within 1e-6 degrees:",
      np.sum(rowwise_difference < 1e-6))
print("Maximum coordinate difference:",
      rowwise_difference.max())

# Check whether the coordinate sets match, irrespective of ordering
distance_matrix = cdist(original_coords, current_coords)
nearest_indices = distance_matrix.argmin(axis=1)
nearest_distances = distance_matrix.min(axis=1)

print("\nCoordinate-set comparison:")
print("Unique nearest matches:",
      len(np.unique(nearest_indices)))
print("Matches within 1e-6 degrees:",
      np.sum(nearest_distances < 1e-6))
print("Maximum nearest-coordinate difference:",
      nearest_distances.max())

Original coordinates: (5040, 2)
Current coordinates: (5040, 2)

Row-wise comparison:
Exact matches: 4187
Matches within 1e-6 degrees: 5040
Maximum coordinate difference: 1.464821375527116e-14

Coordinate-set comparison:
Unique nearest matches: 5040
Matches within 1e-6 degrees: 5040
Maximum nearest-coordinate difference: 1.464821375527116e-14


In [20]:
# Original graph node index -> current grid_nodes.csv index
original_to_current = nearest_indices.copy()

# Verify one-to-one mapping
assert len(np.unique(original_to_current)) == 5040
assert np.all(nearest_distances < 1e-6)

# Create reverse mapping: current index -> original graph index
current_to_original = np.argsort(original_to_current)

print("Mapping verification:")
print("Original nodes:", len(original_to_current))
print("Current nodes:", len(current_to_original))
print("Unique mapped nodes:", len(np.unique(original_to_current)))

print("\nFirst 15 mappings:")
mapping_df = pd.DataFrame({
    "original_graph_index": np.arange(15),
    "current_node_index": original_to_current[:15]
})
display(mapping_df)

print("\nMapping is one-to-one:",
      len(np.unique(original_to_current)) == 5040)

Mapping verification:
Original nodes: 5040
Current nodes: 5040
Unique mapped nodes: 5040

First 15 mappings:


,original_graph_index,current_node_index
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4
5,5,5
6,6,6
7,7,7
8,8,8
9,9,9



Mapping is one-to-one: True


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
from scipy.spatial.distance import cdist

# --------------------------------------------------
# 1. Specify both file paths
# --------------------------------------------------

ORIGINAL_FILE = Path(
    r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final-Year-Project\Notebooks\grid_with_rainfall.csv"
)

CURRENT_FILE = Path(
    r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\spatial\grid_nodes.csv"
)

# --------------------------------------------------
# 2. Load both files
# --------------------------------------------------

original_df = pd.read_csv(ORIGINAL_FILE)
current_df = pd.read_csv(CURRENT_FILE)

print("FILE 1:", ORIGINAL_FILE.name)
print("Shape:", original_df.shape)
print("Columns:", original_df.columns.tolist())

print("\nFILE 2:", CURRENT_FILE.name)
print("Shape:", current_df.shape)
print("Columns:", current_df.columns.tolist())

# --------------------------------------------------
# 3. Extract coordinates from original CSV
# Original format: POINT (longitude latitude)
# --------------------------------------------------

def parse_centroid(value):
    values = re.findall(
        r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?",
        str(value)
    )
    lon, lat = map(float, values)
    return lat, lon

original_coords = np.array([
    parse_centroid(value)
    for value in original_df["centroid"]
])

current_coords = current_df[
    ["latitude", "longitude"]
].to_numpy()

# --------------------------------------------------
# 4. Compare row-by-row
# --------------------------------------------------

row_difference = np.linalg.norm(
    original_coords - current_coords,
    axis=1
)

exact_matches = np.sum(row_difference == 0)
row_matches = np.sum(row_difference < 1e-6)

# --------------------------------------------------
# 5. Compare geographical coordinates irrespective
# of their row positions
# --------------------------------------------------

distance_matrix = cdist(original_coords, current_coords)

nearest_indices = distance_matrix.argmin(axis=1)
nearest_distances = distance_matrix.min(axis=1)

unique_matches = len(np.unique(nearest_indices))
geographical_matches = np.sum(nearest_distances < 1e-6)

# --------------------------------------------------
# 6. Display comparison results
# --------------------------------------------------

print("\n========== COMPARISON RESULTS ==========")

print("\nRow-wise comparison:")
print("Exact matches:", exact_matches)
print("Matches within tolerance:", row_matches)
print("Different row positions:", 5040 - row_matches)

print("\nGeographical comparison:")
print("Unique coordinate matches:", unique_matches)
print("Matches within tolerance:", geographical_matches)
print("Maximum coordinate difference:",
      nearest_distances.max())

print("\nFinal verification:")
print("Same number of nodes:",
      len(original_coords) == len(current_coords))
print("Same geographical node set:",
      geographical_matches == 5040 and unique_matches == 5040)
print("Same row ordering:", row_matches == 5040)

FILE 1: grid_with_rainfall.csv
Shape: (5040, 2)
Columns: ['centroid', 'rainfall']

FILE 2: grid_nodes.csv
Shape: (5040, 2)
Columns: ['latitude', 'longitude']

========== COMPARISON RESULTS ==========

Row-wise comparison:
Exact matches: 4187
Matches within tolerance: 5040
Different row positions: 0

Geographical comparison:
Unique coordinate matches: 5040
Matches within tolerance: 5040
Maximum coordinate difference: 1.464821375527116e-14

Final verification:
Same number of nodes: True
Same geographical node set: True
Same row ordering: True


In [22]:
import torch
import numpy as np
from pathlib import Path

GRAPH_FILE = Path(
    r"D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\Fi Year Project\Final Codes\static_spatial_graph.pt"
)

original_graph = torch.load(
    GRAPH_FILE,
    map_location="cpu",
    weights_only=False
)

edge_index = original_graph.edge_index.numpy()

outgoing_degree = np.bincount(
    edge_index[0],
    minlength=5040
)

print("Graph file:", GRAPH_FILE.name)
print("Number of nodes:", original_graph.num_nodes)
print("Number of edges:", edge_index.shape[1])
print("Minimum node index:", edge_index.min())
print("Maximum node index:", edge_index.max())
print("Minimum outgoing degree:", outgoing_degree.min())
print("Maximum outgoing degree:", outgoing_degree.max())
print("Nodes with exactly 8 neighbours:",
      np.sum(outgoing_degree == 8))
print("Self-loops:", np.sum(edge_index[0] == edge_index[1]))
print("Unique edges:", np.unique(edge_index.T, axis=0).shape[0])

assert original_graph.num_nodes == 5040
assert edge_index.shape[1] == 40320
assert np.all(outgoing_degree == 8)
assert np.sum(edge_index[0] == edge_index[1]) == 0

print("\nAll graph structural checks passed.")

Graph file: static_spatial_graph.pt
Number of nodes: 5040
Number of edges: 40320
Minimum node index: 0
Maximum node index: 5039
Minimum outgoing degree: 8
Maximum outgoing degree: 8
Nodes with exactly 8 neighbours: 5040
Self-loops: 0
Unique edges: 40320

All graph structural checks passed.


C:\Users\rajgm\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch_geometric\data\data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


In [23]:
# Explicitly set the number of nodes
original_graph.num_nodes = 5040

# Save a verified copy
VERIFIED_GRAPH_FILE = (
    SPATIAL_DIR / "static_spatial_graph_verified.pt"
)

torch.save(original_graph, VERIFIED_GRAPH_FILE)

# Reload and verify
verified_graph = torch.load(
    VERIFIED_GRAPH_FILE,
    map_location="cpu",
    weights_only=False
)

print("Verified graph saved successfully.")
print("File:", VERIFIED_GRAPH_FILE)
print("Number of nodes:", verified_graph.num_nodes)
print("Number of edges:", verified_graph.edge_index.shape[1])
print("Graph connectivity preserved:",
      torch.equal(verified_graph.edge_index,
                  original_graph.edge_index))

Verified graph saved successfully.
File: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\spatial\static_spatial_graph_verified.pt
Number of nodes: 5040
Number of edges: 40320
Graph connectivity preserved: True


In [24]:
# Step 14: Calculate training-only monthly climatology

# Extract training-period observations
rainfall_train = rainfall_data[train_mask].copy()
temperature_train = temperature_data[train_mask].copy()

# Extract corresponding training dates
training_dates = dates[train_mask]

# Identify calendar months
training_months = training_dates.month

# Calculate monthly climatology for each node
rainfall_climatology = np.full((12, 5040), np.nan)
temperature_climatology = np.full((12, 5040), np.nan)

for month in range(1, 13):

    month_mask = training_months == month

    rainfall_climatology[month - 1] = np.nanmean(
        rainfall_train[month_mask],
        axis=0
    )

    temperature_climatology[month - 1] = np.nanmean(
        temperature_train[month_mask],
        axis=0
    )

print("Training-only climatology calculated.")

print("\nRainfall climatology shape:",
      rainfall_climatology.shape)

print("Temperature climatology shape:",
      temperature_climatology.shape)

print("\nRainfall nodes with available climatology:",
      np.isfinite(rainfall_climatology).sum())

print("Temperature nodes with available climatology:",
      np.isfinite(temperature_climatology).sum())

C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\3172148908.py:21: RuntimeWarning: Mean of empty slice
  rainfall_climatology[month - 1] = np.nanmean(
C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\3172148908.py:26: RuntimeWarning: Mean of empty slice
  temperature_climatology[month - 1] = np.nanmean(


Training-only climatology calculated.

Rainfall climatology shape: (12, 5040)
Temperature climatology shape: (12, 5040)

Rainfall nodes with available climatology: 56832
Temperature nodes with available climatology: 53712


In [25]:
# Step 15: Validate monthly climatology

for name, climatology in [
    ("Rainfall", rainfall_climatology),
    ("Temperature", temperature_climatology)
]:

    valid_values = climatology[np.isfinite(climatology)]

    print(f"\n{name} climatology")
    print("-" * 35)
    print("Valid values:", len(valid_values))
    print("Missing values:", np.isnan(climatology).sum())
    print("Minimum:", np.min(valid_values))
    print("Maximum:", np.max(valid_values))
    print("Mean:", np.mean(valid_values))
    print("Standard deviation:", np.std(valid_values))

# Physical validity checks
assert np.all(rainfall_climatology[np.isfinite(rainfall_climatology)] >= 0)

assert np.all(np.isfinite(temperature_climatology) |
              np.isnan(temperature_climatology))

print("\nClimatology validation completed.")


Rainfall climatology
-----------------------------------
Valid values: 56832
Missing values: 3648
Minimum: 0.0
Maximum: 55.160682678222656
Mean: 2.986968558035295
Standard deviation: 4.237267251176146

Temperature climatology
-----------------------------------
Valid values: 53712
Missing values: 6768
Minimum: 9.305663108825684
Maximum: 42.57741165161133
Mean: 31.23899013276323
Standard deviation: 5.342588459358085

Climatology validation completed.


In [26]:
# Step 16: Check spatial fallback availability

import torch
import numpy as np
import pandas as pd

# Load verified graph
verified_graph = torch.load(
    SPATIAL_DIR / "static_spatial_graph_verified.pt",
    map_location="cpu",
    weights_only=False
)

edge_index = verified_graph.edge_index.numpy()

# Identify incoming neighbours for each node
neighbours = {
    node: edge_index[1, edge_index[0] == node]
    for node in range(5040)
}

# Check neighbour availability during training
rainfall_node_available = rainfall_train_mask.any(axis=0)
temperature_node_available = temperature_train_mask.any(axis=0)

results = []

for variable, available in [
    ("Rainfall", rainfall_node_available),
    ("Temperature", temperature_node_available)
]:

    for node in np.where(~available)[0]:

        neighbour_nodes = neighbours[node]
        available_neighbours = available[neighbour_nodes].sum()

        results.append({
            "variable": variable,
            "node_index": node,
            "available_neighbours": int(available_neighbours),
            "total_neighbours": len(neighbour_nodes)
        })

fallback_df = pd.DataFrame(results)

print("Spatial fallback assessment")
print("--------------------------------")

for variable in ["Rainfall", "Temperature"]:

    subset = fallback_df[
        fallback_df["variable"] == variable
    ]

    print(f"\n{variable}")
    print("Nodes without training observations:", len(subset))
    print("Mean available neighbours:",
          subset["available_neighbours"].mean())
    print("Nodes with at least one available neighbour:",
          (subset["available_neighbours"] > 0).sum())
    print("Nodes with no available neighbours:",
          (subset["available_neighbours"] == 0).sum())

display(fallback_df.head(10))

Spatial fallback assessment
--------------------------------

Rainfall
Nodes without training observations: 304
Mean available neighbours: 5.128289473684211
Nodes with at least one available neighbour: 277
Nodes with no available neighbours: 27

Temperature
Nodes without training observations: 564
Mean available neighbours: 3.148936170212766
Nodes with at least one available neighbour: 436
Nodes with no available neighbours: 128


,variable,node_index,available_neighbours,total_neighbours
0,Rainfall,0,8,8
1,Rainfall,1,7,8
2,Rainfall,6,8,8
3,Rainfall,45,8,8
4,Rainfall,116,4,8
5,Rainfall,117,4,8
6,Rainfall,118,4,8
7,Rainfall,119,4,8
8,Rainfall,120,4,8
9,Rainfall,121,5,8


In [27]:
# Step 17: Check monthly spatial fallback coverage

monthly_spatial_coverage = []

for variable, climatology in [
    ("Rainfall", rainfall_climatology),
    ("Temperature", temperature_climatology)
]:

    for month in range(12):

        missing_nodes = np.where(
            ~np.isfinite(climatology[month])
        )[0]

        supported = 0

        for node in missing_nodes:

            neighbour_nodes = neighbours[node]

            available = np.isfinite(
                climatology[month, neighbour_nodes]
            )

            if available.any():
                supported += 1

        monthly_spatial_coverage.append({
            "variable": variable,
            "month": month + 1,
            "nodes_without_climatology": len(missing_nodes),
            "nodes_with_spatial_support": supported,
            "nodes_without_spatial_support":
                len(missing_nodes) - supported
        })

monthly_spatial_df = pd.DataFrame(
    monthly_spatial_coverage
)

display(
    monthly_spatial_df.groupby("variable")[
        [
            "nodes_without_climatology",
            "nodes_with_spatial_support",
            "nodes_without_spatial_support"
        ]
    ].mean().round(2)
)

print("\nMonthly details:")
display(monthly_spatial_df)

,nodes_without_climatology,nodes_with_spatial_support,nodes_without_spatial_support
variable,,,
Rainfall,304.0,277.0,27.0
Temperature,564.0,436.0,128.0



Monthly details:


,variable,month,nodes_without_climatology,nodes_with_spatial_support,nodes_without_spatial_support
0,Rainfall,1,304,277,27
1,Rainfall,2,304,277,27
2,Rainfall,3,304,277,27
3,Rainfall,4,304,277,27
4,Rainfall,5,304,277,27
5,Rainfall,6,304,277,27
6,Rainfall,7,304,277,27
7,Rainfall,8,304,277,27
8,Rainfall,9,304,277,27
9,Rainfall,10,304,277,27


In [28]:
# Step 18: Define leakage-safe imputation function

def impute_climate_variable(
    data,
    date_index,
    climatology,
    neighbour_dict,
    max_forward_gap=3
):
    """
    Impute missing input observations using:
    1. Forward fill up to max_forward_gap days.
    2. Training-only monthly climatology.
    3. Spatial-neighbour monthly climatology.

    Original valid observations are never overwritten.
    """

    data = data.astype(np.float32).copy()

    original_mask = np.isfinite(data)
    imputed_mask = np.zeros(data.shape, dtype=bool)

    output = data.copy()

    # Forward fill only within the supplied time segment
    for node in range(data.shape[1]):

        last_value = np.nan
        gap_length = 0

        for t in range(data.shape[0]):

            if original_mask[t, node]:
                last_value = data[t, node]
                gap_length = 0

            else:
                gap_length += 1

                if (
                    np.isfinite(last_value)
                    and gap_length <= max_forward_gap
                ):
                    output[t, node] = last_value
                    imputed_mask[t, node] = True

    # Monthly climatology and spatial fallback
    for t, date in enumerate(date_index):

        month_index = date.month - 1

        missing_nodes = np.where(
            ~np.isfinite(output[t])
        )[0]

        for node in missing_nodes:

            value = climatology[month_index, node]

            if np.isfinite(value):
                output[t, node] = value
                imputed_mask[t, node] = True

            else:
                neighbour_nodes = neighbour_dict[node]

                neighbour_values = climatology[
                    month_index, neighbour_nodes
                ]

                valid_neighbours = neighbour_values[
                    np.isfinite(neighbour_values)
                ]

                if len(valid_neighbours) > 0:
                    output[t, node] = np.mean(valid_neighbours)
                    imputed_mask[t, node] = True

    return output, original_mask, imputed_mask

In [29]:
# Verify function definition
print("Imputation function defined:", callable(impute_climate_variable))
print("Maximum forward-fill gap:", 3, "days")

Imputation function defined: True
Maximum forward-fill gap: 3 days


In [30]:
# Step 19: Apply imputation to training data

rainfall_train_imputed, rainfall_train_original_mask, rainfall_train_imputed_mask = (
    impute_climate_variable(
        rainfall_train,
        training_dates,
        rainfall_climatology,
        neighbours,
        max_forward_gap=3
    )
)

temperature_train_imputed, temperature_train_original_mask, temperature_train_imputed_mask = (
    impute_climate_variable(
        temperature_train,
        training_dates,
        temperature_climatology,
        neighbours,
        max_forward_gap=3
    )
)

print("Rainfall imputed shape:", rainfall_train_imputed.shape)
print("Temperature imputed shape:", temperature_train_imputed.shape)

print("\nRainfall:")
print("Original valid observations:", rainfall_train_original_mask.sum())
print("Imputed observations:", rainfall_train_imputed_mask.sum())
print("Remaining missing values:", np.isnan(rainfall_train_imputed).sum())

print("\nTemperature:")
print("Original valid observations:", temperature_train_original_mask.sum())
print("Imputed observations:", temperature_train_imputed_mask.sum())
print("Remaining missing values:", np.isnan(temperature_train_imputed).sum())

# Verify that original observations were not modified
assert np.array_equal(
    rainfall_train_imputed[rainfall_train_original_mask],
    rainfall_train[rainfall_train_original_mask]
)

assert np.array_equal(
    temperature_train_imputed[temperature_train_original_mask],
    temperature_train[temperature_train_original_mask]
)

# Verify imputation masks
assert not np.any(rainfall_train_imputed_mask & rainfall_train_original_mask)
assert not np.any(temperature_train_imputed_mask & temperature_train_original_mask)

print("\nOriginal observations preserved: True")
print("Imputation mask checks passed: True")

Rainfall imputed shape: (6575, 5040)
Temperature imputed shape: (6575, 5040)

Rainfall:
Original valid observations: 31139200
Imputed observations: 1821275
Remaining missing values: 177525

Temperature:
Original valid observations: 29268202
Imputed observations: 3028198
Remaining missing values: 841600

Original observations preserved: True
Imputation mask checks passed: True


In [31]:
# Step 20: Define causal, leakage-safe imputation

def impute_climate_causally(
    data,
    date_index,
    neighbour_dict,
    edge_index,
    max_forward_gap=3
):
    data = data.astype(np.float32).copy()

    n_days, n_nodes = data.shape
    original_mask = np.isfinite(data)
    output = data.copy()
    imputed_mask = np.zeros(data.shape, dtype=bool)

    # Step 1: Forward-fill using only previous observations
    for node in range(n_nodes):
        last_value = np.nan
        gap_length = 0

        for t in range(n_days):
            if original_mask[t, node]:
                last_value = data[t, node]
                gap_length = 0
            else:
                gap_length += 1

                if np.isfinite(last_value) and gap_length <= max_forward_gap:
                    output[t, node] = last_value
                    imputed_mask[t, node] = True

    # Step 2: Prepare historical monthly statistics
    historical_sum = np.zeros((12, n_nodes), dtype=np.float64)
    historical_count = np.zeros((12, n_nodes), dtype=np.int32)

    source_nodes = edge_index[0]
    target_nodes = edge_index[1]

    # Step 3: Process dates chronologically
    for t, date in enumerate(date_index):

        month_index = date.month - 1

        # Historical means use observations strictly before this date
        counts = historical_count[month_index]
        valid_history = counts > 0

        historical_mean = np.full(n_nodes, np.nan, dtype=np.float64)
        historical_mean[valid_history] = (
            historical_sum[month_index, valid_history]
            / counts[valid_history]
        )

        missing_nodes = ~np.isfinite(output[t])

        # Step 4: Use the node's own historical monthly mean
        own_fill = missing_nodes & valid_history
        output[t, own_fill] = historical_mean[own_fill]
        imputed_mask[t, own_fill] = True

        # Step 5: Spatial fallback for nodes without historical observations
        remaining = ~np.isfinite(output[t])

        if np.any(remaining):
            valid_neighbours = np.isfinite(historical_mean[target_nodes])

            neighbour_sum = np.bincount(
                source_nodes,
                weights=np.where(
                    valid_neighbours,
                    historical_mean[target_nodes],
                    0.0
                ),
                minlength=n_nodes
            )

            neighbour_count = np.bincount(
                source_nodes,
                weights=valid_neighbours.astype(np.float64),
                minlength=n_nodes
            )

            spatial_fill = remaining & (neighbour_count > 0)

            output[t, spatial_fill] = (
                neighbour_sum[spatial_fill] /
                neighbour_count[spatial_fill]
            )

            imputed_mask[t, spatial_fill] = True

        # Step 6: Update statistics using original observations only
        observed_today = original_mask[t]

        historical_sum[month_index, observed_today] += (
            data[t, observed_today]
        )

        historical_count[month_index, observed_today] += 1

    return output, original_mask, imputed_mask


print("Causal imputation function defined:", callable(impute_climate_causally))
print("Future observations used for imputation: No")
print("Imputed values included in historical statistics: No")

Causal imputation function defined: True
Future observations used for imputation: No
Imputed values included in historical statistics: No


In [32]:
# Step 21: Apply causal imputation to training data

rainfall_train_causal, rainfall_original_mask_causal, rainfall_imputed_mask_causal = (
    impute_climate_causally(
        rainfall_train,
        training_dates,
        neighbours,
        edge_index,
        max_forward_gap=3
    )
)

temperature_train_causal, temperature_original_mask_causal, temperature_imputed_mask_causal = (
    impute_climate_causally(
        temperature_train,
        training_dates,
        neighbours,
        edge_index,
        max_forward_gap=3
    )
)

print("RAINFALL")
print("Original valid:", rainfall_original_mask_causal.sum())
print("Imputed:", rainfall_imputed_mask_causal.sum())
print("Remaining missing:", np.isnan(rainfall_train_causal).sum())

print("\nTEMPERATURE")
print("Original valid:", temperature_original_mask_causal.sum())
print("Imputed:", temperature_imputed_mask_causal.sum())
print("Remaining missing:", np.isnan(temperature_train_causal).sum())

# Verify original observations remain unchanged
assert np.array_equal(
    rainfall_train_causal[rainfall_original_mask_causal],
    rainfall_train[rainfall_original_mask_causal]
)

assert np.array_equal(
    temperature_train_causal[temperature_original_mask_causal],
    temperature_train[temperature_original_mask_causal]
)

# Verify imputed values do not overlap original observations
assert not np.any(
    rainfall_original_mask_causal & rainfall_imputed_mask_causal
)

assert not np.any(
    temperature_original_mask_causal & temperature_imputed_mask_causal
)

print("\nOriginal observations preserved: True")
print("Missingness mask checks passed: True")

RAINFALL
Original valid: 31139200
Imputed: 1817951
Remaining missing: 180849

TEMPERATURE
Original valid: 29268202
Imputed: 2975442
Remaining missing: 894356

Original observations preserved: True
Missingness mask checks passed: True


In [33]:
# Step 22: Analyse remaining missing values

rainfall_remaining_mask = ~np.isfinite(rainfall_train_causal)
temperature_remaining_mask = ~np.isfinite(temperature_train_causal)

rainfall_remaining_by_node = rainfall_remaining_mask.sum(axis=0)
temperature_remaining_by_node = temperature_remaining_mask.sum(axis=0)

rainfall_remaining_by_day = rainfall_remaining_mask.sum(axis=1)
temperature_remaining_by_day = temperature_remaining_mask.sum(axis=1)

print("RAINFALL")
print("Nodes with remaining missing values:",
      np.sum(rainfall_remaining_by_node > 0))
print("Nodes completely missing after imputation:",
      np.sum(rainfall_remaining_by_node == len(training_dates)))
print("Maximum missing values in one node:",
      rainfall_remaining_by_node.max())
print("Days with at least one missing value:",
      np.sum(rainfall_remaining_by_day > 0))
print("Maximum missing nodes on one day:",
      rainfall_remaining_by_day.max())

print("\nTEMPERATURE")
print("Nodes with remaining missing values:",
      np.sum(temperature_remaining_by_node > 0))
print("Nodes completely missing after imputation:",
      np.sum(temperature_remaining_by_node == len(training_dates)))
print("Maximum missing values in one node:",
      temperature_remaining_by_node.max())
print("Days with at least one missing value:",
      np.sum(temperature_remaining_by_day > 0))
print("Maximum missing nodes on one day:",
      temperature_remaining_by_day.max())

# Verify that all remaining missing values were originally missing
assert np.all(~rainfall_original_mask_causal[rainfall_remaining_mask])
assert np.all(~temperature_original_mask_causal[temperature_remaining_mask])

print("\nRemaining missing values were not original observations: True")

RAINFALL
Nodes with remaining missing values: 304
Nodes completely missing after imputation: 27
Maximum missing values in one node: 6575
Days with at least one missing value: 6575
Maximum missing nodes on one day: 304

TEMPERATURE
Nodes with remaining missing values: 613
Nodes completely missing after imputation: 128
Maximum missing values in one node: 6575
Days with at least one missing value: 6575
Maximum missing nodes on one day: 613

Remaining missing values were not original observations: True


In [34]:
# Step 23: Quantify genuine supervised training observations

joint_train_original_mask = (
    rainfall_original_mask_causal &
    temperature_original_mask_causal
)

rainfall_only_train_mask = (
    rainfall_original_mask_causal &
    ~temperature_original_mask_causal
)

temperature_only_train_mask = (
    ~rainfall_original_mask_causal &
    temperature_original_mask_causal
)

print("TRAINING TARGET AVAILABILITY")
print("--------------------------------")

print("Total possible observations:",
      joint_train_original_mask.size)

print("Jointly observed rainfall-temperature values:",
      joint_train_original_mask.sum())

print("Rainfall observed only:",
      rainfall_only_train_mask.sum())

print("Temperature observed only:",
      temperature_only_train_mask.sum())

print("Both variables missing:",
      (~rainfall_original_mask_causal &
       ~temperature_original_mask_causal).sum())

print("\nJoint target availability (%):",
      round(100 * joint_train_original_mask.mean(), 3))

print("Nodes with at least one joint target:",
      joint_train_original_mask.any(axis=0).sum())

print("Nodes with no joint targets:",
      (~joint_train_original_mask.any(axis=0)).sum())

# Verify that target masks are based on original observations
assert np.array_equal(
    joint_train_original_mask,
    np.isfinite(rainfall_train) & np.isfinite(temperature_train)
)

print("\nOriginal target-mask verification: Passed")

TRAINING TARGET AVAILABILITY
--------------------------------
Total possible observations: 33138000
Jointly observed rainfall-temperature values: 28551469
Rainfall observed only: 2587731
Temperature observed only: 716733
Both variables missing: 1282067

Joint target availability (%): 86.159
Nodes with at least one joint target: 4365
Nodes with no joint targets: 675

Original target-mask verification: Passed


In [35]:
# Step 24: Calculate leakage-safe normalization statistics

rainfall_observed_values = rainfall_train[
    rainfall_original_mask_causal
]

temperature_observed_values = temperature_train[
    temperature_original_mask_causal
]

rainfall_mean = np.mean(rainfall_observed_values)
rainfall_std = np.std(rainfall_observed_values)

temperature_mean = np.mean(temperature_observed_values)
temperature_std = np.std(temperature_observed_values)

print("TRAINING NORMALIZATION STATISTICS")
print("----------------------------------")

print("Rainfall mean:", rainfall_mean)
print("Rainfall standard deviation:", rainfall_std)

print("\nTemperature mean:", temperature_mean)
print("Temperature standard deviation:", temperature_std)

# Verify statistics are finite and standard deviations are positive
assert np.isfinite(rainfall_mean) and rainfall_std > 0
assert np.isfinite(temperature_mean) and temperature_std > 0

print("\nNormalization statistics verified: True")
print("Statistics calculated from original observations only: True")

TRAINING NORMALIZATION STATISTICS
----------------------------------
Rainfall mean: 3.0009441
Rainfall standard deviation: 10.183425

Temperature mean: 31.261608
Temperature standard deviation: 5.7920213

Normalization statistics verified: True
Statistics calculated from original observations only: True


In [36]:
# Step 25: Prepare normalized training input features

# Normalize using training-only statistics
rainfall_normalized = (
    rainfall_train_causal - rainfall_mean
) / rainfall_std

temperature_normalized = (
    temperature_train_causal - temperature_mean
) / temperature_std

# Replace unresolved missing values with zero in standardized space
rainfall_normalized = np.nan_to_num(
    rainfall_normalized, nan=0.0
).astype(np.float32)

temperature_normalized = np.nan_to_num(
    temperature_normalized, nan=0.0
).astype(np.float32)

# Create observation and imputation indicators
rainfall_observed_indicator = (
    rainfall_original_mask_causal.astype(np.float32)
)

temperature_observed_indicator = (
    temperature_original_mask_causal.astype(np.float32)
)

rainfall_imputation_indicator = (
    rainfall_imputed_mask_causal.astype(np.float32)
)

temperature_imputation_indicator = (
    temperature_imputed_mask_causal.astype(np.float32)
)

# Stack six input channels: (time, node, features)
X_train = np.stack(
    [
        rainfall_normalized,
        temperature_normalized,
        rainfall_observed_indicator,
        temperature_observed_indicator,
        rainfall_imputation_indicator,
        temperature_imputation_indicator
    ],
    axis=-1
).astype(np.float32)

print("Training input shape:", X_train.shape)
print("Expected shape:", (6575, 5040, 6))
print("Data type:", X_train.dtype)
print("All input values finite:", np.isfinite(X_train).all())

print("\nFeature channels:")
print("0: Normalized rainfall")
print("1: Normalized maximum temperature")
print("2: Original rainfall observation indicator")
print("3: Original temperature observation indicator")
print("4: Rainfall imputation indicator")
print("5: Temperature imputation indicator")

# Validation checks
assert X_train.shape == (6575, 5040, 6)
assert np.isfinite(X_train).all()

assert np.all(
    X_train[..., 2] + X_train[..., 4] <= 1
)

assert np.all(
    X_train[..., 3] + X_train[..., 5] <= 1
)

print("\nTraining input preparation: Passed")

Training input shape: (6575, 5040, 6)
Expected shape: (6575, 5040, 6)
Data type: float32
All input values finite: True

Feature channels:
0: Normalized rainfall
1: Normalized maximum temperature
2: Original rainfall observation indicator
3: Original temperature observation indicator
4: Rainfall imputation indicator
5: Temperature imputation indicator

Training input preparation: Passed


In [37]:
# Step 26: Prepare training targets and original observation masks

Y_train = np.stack(
    [rainfall_train, temperature_train],
    axis=-1
).astype(np.float32)

Y_train_mask = np.stack(
    [
        rainfall_original_mask_causal,
        temperature_original_mask_causal
    ],
    axis=-1
).astype(bool)

print("Training target shape:", Y_train.shape)
print("Target mask shape:", Y_train_mask.shape)

print("\nRainfall genuine targets:",
      Y_train_mask[..., 0].sum())

print("Temperature genuine targets:",
      Y_train_mask[..., 1].sum())

print("Rainfall missing targets:",
      (~Y_train_mask[..., 0]).sum())

print("Temperature missing targets:",
      (~Y_train_mask[..., 1]).sum())

# Verify missing targets remain missing
assert np.isnan(Y_train[..., 0][~Y_train_mask[..., 0]]).all()
assert np.isnan(Y_train[..., 1][~Y_train_mask[..., 1]]).all()

# Verify observed targets are unchanged
assert np.array_equal(
    Y_train[..., 0][Y_train_mask[..., 0]],
    rainfall_train[Y_train_mask[..., 0]]
)

assert np.array_equal(
    Y_train[..., 1][Y_train_mask[..., 1]],
    temperature_train[Y_train_mask[..., 1]]
)

print("\nOriginal target values preserved: True")
print("Missing targets remain excluded: True")

Training target shape: (6575, 5040, 2)
Target mask shape: (6575, 5040, 2)

Rainfall genuine targets: 31139200
Temperature genuine targets: 29268202
Rainfall missing targets: 1998800
Temperature missing targets: 3869798

Original target values preserved: True
Missing targets remain excluded: True


In [38]:
# Step 27: Extract validation and test data

rainfall_validation = rainfall_data[validation_mask].copy()
temperature_validation = temperature_data[validation_mask].copy()

rainfall_test = rainfall_data[test_mask].copy()
temperature_test = temperature_data[test_mask].copy()

validation_dates = dates[validation_mask]
test_dates = dates[test_mask]

print("Validation rainfall:", rainfall_validation.shape)
print("Validation temperature:", temperature_validation.shape)

print("\nTest rainfall:", rainfall_test.shape)
print("Test temperature:", temperature_test.shape)

print("\nValidation period:",
      validation_dates[0], "to", validation_dates[-1])

print("Test period:",
      test_dates[0], "to", test_dates[-1])

assert rainfall_validation.shape == (2191, 5040)
assert temperature_validation.shape == (2191, 5040)
assert rainfall_test.shape == (731, 5040)
assert temperature_test.shape == (731, 5040)

print("\nTemporal split verification: Passed")

Validation rainfall: (2191, 5040)
Validation temperature: (2191, 5040)

Test rainfall: (731, 5040)
Test temperature: (731, 5040)

Validation period: 2018-01-01 00:00:00 to 2023-12-31 00:00:00
Test period: 2024-01-01 00:00:00 to 2025-12-31 00:00:00

Temporal split verification: Passed


In [39]:
# Step 28: Define imputation for validation and test periods

def get_previous_observation_state(previous_data, max_gap=3):
    """Get last original observation and trailing missing-gap length per node."""
    n_nodes = previous_data.shape[1]

    last_values = np.full(n_nodes, np.nan, dtype=np.float32)
    gap_lengths = np.zeros(n_nodes, dtype=np.int32)

    for node in range(n_nodes):
        observed_indices = np.where(
            np.isfinite(previous_data[:, node])
        )[0]

        if len(observed_indices) > 0:
            last_idx = observed_indices[-1]
            last_values[node] = previous_data[last_idx, node]
            gap_lengths[node] = (
                previous_data.shape[0] - 1 - last_idx
            )
        else:
            gap_lengths[node] = max_gap + 1

    return last_values, gap_lengths


def impute_evaluation_period(
    data,
    date_index,
    climatology,
    neighbour_dict,
    previous_data,
    max_forward_gap=3
):
    data = data.astype(np.float32).copy()

    original_mask = np.isfinite(data)
    output = data.copy()
    imputed_mask = np.zeros(data.shape, dtype=bool)

    last_values, gap_lengths = get_previous_observation_state(
        previous_data,
        max_forward_gap
    )

    for t, date in enumerate(date_index):

        # Causal forward-fill, including the preceding period
        for node in range(data.shape[1]):
            if original_mask[t, node]:
                last_values[node] = data[t, node]
                gap_lengths[node] = 0
            else:
                gap_lengths[node] += 1

                if (
                    np.isfinite(last_values[node])
                    and gap_lengths[node] <= max_forward_gap
                ):
                    output[t, node] = last_values[node]
                    imputed_mask[t, node] = True

        # Training-only monthly climatology
        month_index = date.month - 1
        missing_nodes = np.where(~np.isfinite(output[t]))[0]

        for node in missing_nodes:
            value = climatology[month_index, node]

            if np.isfinite(value):
                output[t, node] = value
                imputed_mask[t, node] = True
            else:
                neighbour_nodes = neighbour_dict[node]
                neighbour_values = climatology[
                    month_index, neighbour_nodes
                ]
                valid_values = neighbour_values[
                    np.isfinite(neighbour_values)
                ]

                if len(valid_values) > 0:
                    output[t, node] = np.mean(valid_values)
                    imputed_mask[t, node] = True

    return output, original_mask, imputed_mask


print("Evaluation imputation function defined:",
      callable(impute_evaluation_period))

Evaluation imputation function defined: True


In [40]:
# Step 29: Causal imputation of validation data

rainfall_validation_imputed, rainfall_validation_original_mask, rainfall_validation_imputed_mask = (
    impute_evaluation_period(
        rainfall_validation,
        validation_dates,
        rainfall_climatology,
        neighbours,
        rainfall_train,
        max_forward_gap=3
    )
)

temperature_validation_imputed, temperature_validation_original_mask, temperature_validation_imputed_mask = (
    impute_evaluation_period(
        temperature_validation,
        validation_dates,
        temperature_climatology,
        neighbours,
        temperature_train,
        max_forward_gap=3
    )
)

print("RAINFALL")
print("Original valid:", rainfall_validation_original_mask.sum())
print("Imputed:", rainfall_validation_imputed_mask.sum())
print("Remaining missing:", np.isnan(rainfall_validation_imputed).sum())

print("\nTEMPERATURE")
print("Original valid:", temperature_validation_original_mask.sum())
print("Imputed:", temperature_validation_imputed_mask.sum())
print("Remaining missing:", np.isnan(temperature_validation_imputed).sum())

# Verify original observations are unchanged
assert np.array_equal(
    rainfall_validation_imputed[rainfall_validation_original_mask],
    rainfall_validation[rainfall_validation_original_mask]
)

assert np.array_equal(
    temperature_validation_imputed[temperature_validation_original_mask],
    temperature_validation[temperature_validation_original_mask]
)

print("\nOriginal validation observations preserved: True")

RAINFALL
Original valid: 10376508
Imputed: 606975
Remaining missing: 59157

TEMPERATURE
Original valid: 9786285
Imputed: 975907
Remaining missing: 280448

Original validation observations preserved: True


In [41]:
# Step 30: Causal imputation of test data

rainfall_test_imputed, rainfall_test_original_mask, rainfall_test_imputed_mask = (
    impute_evaluation_period(
        rainfall_test,
        test_dates,
        rainfall_climatology,
        neighbours,
        rainfall_validation,
        max_forward_gap=3
    )
)

temperature_test_imputed, temperature_test_original_mask, temperature_test_imputed_mask = (
    impute_evaluation_period(
        temperature_test,
        test_dates,
        temperature_climatology,
        neighbours,
        temperature_validation,
        max_forward_gap=3
    )
)

print("RAINFALL")
print("Original valid:", rainfall_test_original_mask.sum())
print("Imputed:", rainfall_test_imputed_mask.sum())
print("Remaining missing:", np.isnan(rainfall_test_imputed).sum())

print("\nTEMPERATURE")
print("Original valid:", temperature_test_original_mask.sum())
print("Imputed:", temperature_test_imputed_mask.sum())
print("Remaining missing:", np.isnan(temperature_test_imputed).sum())

# Verify original observations are unchanged
assert np.array_equal(
    rainfall_test_imputed[rainfall_test_original_mask],
    rainfall_test[rainfall_test_original_mask]
)

assert np.array_equal(
    temperature_test_imputed[temperature_test_original_mask],
    temperature_test[temperature_test_original_mask]
)

print("\nOriginal test observations preserved: True")

RAINFALL
Original valid: 3462016
Imputed: 202487
Remaining missing: 19737

TEMPERATURE
Original valid: 3269916
Imputed: 320756
Remaining missing: 93568

Original test observations preserved: True


In [42]:
# Step 31: Prepare normalized validation and test tensors

def prepare_model_tensors(
    rainfall_imputed,
    temperature_imputed,
    rainfall_original_mask,
    temperature_original_mask,
    rainfall_imputed_mask,
    temperature_imputed_mask,
    rainfall_raw,
    temperature_raw
):
    # Normalize using training-only statistics
    rainfall_norm = (
        rainfall_imputed - rainfall_mean
    ) / rainfall_std

    temperature_norm = (
        temperature_imputed - temperature_mean
    ) / temperature_std

    # Replace unresolved missing inputs with zero
    rainfall_norm = np.nan_to_num(
        rainfall_norm, nan=0.0
    ).astype(np.float32)

    temperature_norm = np.nan_to_num(
        temperature_norm, nan=0.0
    ).astype(np.float32)

    # Six input channels
    X = np.stack([
        rainfall_norm,
        temperature_norm,
        rainfall_original_mask.astype(np.float32),
        temperature_original_mask.astype(np.float32),
        rainfall_imputed_mask.astype(np.float32),
        temperature_imputed_mask.astype(np.float32)
    ], axis=-1).astype(np.float32)

    # Original observations only as targets
    Y = np.stack([
        rainfall_raw,
        temperature_raw
    ], axis=-1).astype(np.float32)

    Y_mask = np.stack([
        rainfall_original_mask,
        temperature_original_mask
    ], axis=-1).astype(bool)

    return X, Y, Y_mask


X_validation, Y_validation, Y_validation_mask = prepare_model_tensors(
    rainfall_validation_imputed,
    temperature_validation_imputed,
    rainfall_validation_original_mask,
    temperature_validation_original_mask,
    rainfall_validation_imputed_mask,
    temperature_validation_imputed_mask,
    rainfall_validation,
    temperature_validation
)

X_test, Y_test, Y_test_mask = prepare_model_tensors(
    rainfall_test_imputed,
    temperature_test_imputed,
    rainfall_test_original_mask,
    temperature_test_original_mask,
    rainfall_test_imputed_mask,
    temperature_test_imputed_mask,
    rainfall_test,
    temperature_test
)

print("VALIDATION")
print("X:", X_validation.shape)
print("Y:", Y_validation.shape)
print("Mask:", Y_validation_mask.shape)
print("All input values finite:", np.isfinite(X_validation).all())

print("\nTEST")
print("X:", X_test.shape)
print("Y:", Y_test.shape)
print("Mask:", Y_test_mask.shape)
print("All input values finite:", np.isfinite(X_test).all())

# Structural checks
assert X_validation.shape == (2191, 5040, 6)
assert Y_validation.shape == (2191, 5040, 2)

assert X_test.shape == (731, 5040, 6)
assert Y_test.shape == (731, 5040, 2)

assert np.isfinite(X_validation).all()
assert np.isfinite(X_test).all()

# Verify targets are still original observations
assert np.array_equal(
    Y_validation_mask[..., 0],
    np.isfinite(rainfall_validation)
)

assert np.array_equal(
    Y_test_mask[..., 1],
    np.isfinite(temperature_test)
)

print("\nValidation and test tensor checks: Passed")

VALIDATION
X: (2191, 5040, 6)
Y: (2191, 5040, 2)
Mask: (2191, 5040, 2)
All input values finite: True

TEST
X: (731, 5040, 6)
Y: (731, 5040, 2)
Mask: (731, 5040, 2)
All input values finite: True

Validation and test tensor checks: Passed


In [43]:
# Step 32: Final model-ready dataset integrity check

datasets = {
    "Training": (X_train, Y_train, Y_train_mask),
    "Validation": (X_validation, Y_validation, Y_validation_mask),
    "Testing": (X_test, Y_test, Y_test_mask)
}

print("FINAL DATASET INTEGRITY REPORT")
print("=" * 55)

for name, (X, Y, mask) in datasets.items():

    print(f"\n{name}")
    print("Input shape:", X.shape)
    print("Target shape:", Y.shape)
    print("Mask shape:", mask.shape)

    print("Input finite:", np.isfinite(X).all())
    print("Target mask consistency:",
          np.array_equal(mask, np.isfinite(Y)))

    print("Rainfall valid targets:", mask[..., 0].sum())
    print("Temperature valid targets:", mask[..., 1].sum())

    assert X.shape[:2] == Y.shape[:2]
    assert Y.shape == mask.shape
    assert X.shape[-1] == 6
    assert Y.shape[-1] == 2
    assert np.isfinite(X).all()
    assert np.array_equal(mask, np.isfinite(Y))

# Verify original graph
print("\nGRAPH")
print("Nodes:", verified_graph.num_nodes)
print("Edges:", verified_graph.edge_index.shape[1])

assert verified_graph.num_nodes == 5040
assert verified_graph.edge_index.shape[1] == 40320

print("\nAll final integrity checks passed.")

FINAL DATASET INTEGRITY REPORT

Training
Input shape: (6575, 5040, 6)
Target shape: (6575, 5040, 2)
Mask shape: (6575, 5040, 2)
Input finite: True
Target mask consistency: True
Rainfall valid targets: 31139200
Temperature valid targets: 29268202

Validation
Input shape: (2191, 5040, 6)
Target shape: (2191, 5040, 2)
Mask shape: (2191, 5040, 2)
Input finite: True
Target mask consistency: True
Rainfall valid targets: 10376508
Temperature valid targets: 9786285

Testing
Input shape: (731, 5040, 6)
Target shape: (731, 5040, 2)
Mask shape: (731, 5040, 2)
Input finite: True
Target mask consistency: True
Rainfall valid targets: 3462016
Temperature valid targets: 3269916

GRAPH
Nodes: 5040
Edges: 40320

All final integrity checks passed.


In [44]:
# Step 33: Save model-ready datasets

import json

MODEL_DATA_DIR = PROJECT_DIR / "data" / "processed" / "model_ready"
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Save training dataset
np.savez_compressed(
    MODEL_DATA_DIR / "training_data.npz",
    X=X_train,
    Y=Y_train,
    Y_mask=Y_train_mask,
    dates=training_dates.astype(str)
)

# Save validation dataset
np.savez_compressed(
    MODEL_DATA_DIR / "validation_data.npz",
    X=X_validation,
    Y=Y_validation,
    Y_mask=Y_validation_mask,
    dates=validation_dates.astype(str)
)

# Save test dataset
np.savez_compressed(
    MODEL_DATA_DIR / "test_data.npz",
    X=X_test,
    Y=Y_test,
    Y_mask=Y_test_mask,
    dates=test_dates.astype(str)
)

# Save spatial coordinates
nodes_df.to_csv(
    MODEL_DATA_DIR / "spatial_nodes.csv",
    index=False
)

# Save normalization parameters and feature definitions
metadata = {
    "training_period": "2000-2017",
    "validation_period": "2018-2023",
    "test_period": "2024-2025",
    "number_of_nodes": 5040,
    "number_of_input_features": 6,
    "target_variables": ["rainfall", "temperature"],
    "input_features": [
        "normalized_rainfall",
        "normalized_temperature",
        "rainfall_observed_indicator",
        "temperature_observed_indicator",
        "rainfall_imputation_indicator",
        "temperature_imputation_indicator"
    ],
    "normalization": {
        "rainfall_mean": float(rainfall_mean),
        "rainfall_std": float(rainfall_std),
        "temperature_mean": float(temperature_mean),
        "temperature_std": float(temperature_std)
    },
    "imputation": {
        "forward_fill_max_gap_days": 3,
        "training_method": "causal historical monthly mean with spatial-neighbour fallback",
        "evaluation_method": "training-derived monthly climatology with causal forward-fill",
        "target_imputation": False
    },
    "graph": {
        "nodes": 5040,
        "directed_edges": 40320,
        "method": "Haversine k-nearest neighbours",
        "k": 8
    }
}

with open(
    MODEL_DATA_DIR / "dataset_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=4)

print("Saved files:")
for file in sorted(MODEL_DATA_DIR.iterdir()):
    print(file.name, "-", round(file.stat().st_size / (1024**2), 2), "MB")

print("\nModel-ready dataset directory:")
print(MODEL_DATA_DIR)

Saved files:
dataset_metadata.json - 0.0 MB
spatial_nodes.csv - 0.11 MB
test_data.npz - 37.53 MB
training_data.npz - 331.12 MB
validation_data.npz - 112.23 MB

Model-ready dataset directory:
D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\processed\model_ready


In [45]:
import numpy as np
import json
import pandas as pd

MODEL_DATA_DIR = PROJECT_DIR / "data" / "processed" / "model_ready"

# Load metadata
with open(MODEL_DATA_DIR / "dataset_metadata.json", "r") as f:
    saved_metadata = json.load(f)

# Reload datasets
train_saved = np.load(
    MODEL_DATA_DIR / "training_data.npz", allow_pickle=True
)
val_saved = np.load(
    MODEL_DATA_DIR / "validation_data.npz", allow_pickle=True
)
test_saved = np.load(
    MODEL_DATA_DIR / "test_data.npz", allow_pickle=True
)

datasets = {
    "Training": train_saved,
    "Validation": val_saved,
    "Testing": test_saved
}

for name, data in datasets.items():
    X = data["X"]
    Y = data["Y"]
    Y_mask = data["Y_mask"]
    dates_saved = pd.to_datetime(data["dates"].astype(str))

    print(f"\n{name} dataset")
    print("-" * 45)
    print("X shape:", X.shape)
    print("Y shape:", Y.shape)
    print("Mask shape:", Y_mask.shape)
    print("Date range:", dates_saved.min(), "to", dates_saved.max())
    print("Input contains NaN:", np.isnan(X).any())
    print("Input contains Inf:", np.isinf(X).any())
    print("Mask matches target validity:",
          np.array_equal(Y_mask, np.isfinite(Y)))
    print("Valid rainfall targets:", Y_mask[:, :, 0].sum())
    print("Valid temperature targets:", Y_mask[:, :, 1].sum())

# Verify metadata and spatial nodes
saved_nodes = pd.read_csv(MODEL_DATA_DIR / "spatial_nodes.csv")

print("\nMetadata and spatial checks")
print("-" * 45)
print("Number of nodes in metadata:",
      saved_metadata["number_of_nodes"])
print("Spatial nodes shape:", saved_nodes.shape)
print("Input features:", saved_metadata["number_of_input_features"])
print("Graph configuration:", saved_metadata["graph"])

assert saved_nodes.shape[0] == 5040
assert train_saved["X"].shape == (6575, 5040, 6)
assert val_saved["X"].shape == (2191, 5040, 6)
assert test_saved["X"].shape == (731, 5040, 6)

print("\nAll saved dataset checks completed.")


Training dataset
---------------------------------------------
X shape: (6575, 5040, 6)
Y shape: (6575, 5040, 2)
Mask shape: (6575, 5040, 2)
Date range: 2000-01-01 00:00:00 to 2017-12-31 00:00:00
Input contains NaN: False
Input contains Inf: False
Mask matches target validity: True
Valid rainfall targets: 31139200
Valid temperature targets: 29268202

Validation dataset
---------------------------------------------
X shape: (2191, 5040, 6)
Y shape: (2191, 5040, 2)
Mask shape: (2191, 5040, 2)
Date range: 2018-01-01 00:00:00 to 2023-12-31 00:00:00
Input contains NaN: False
Input contains Inf: False
Mask matches target validity: True
Valid rainfall targets: 10376508
Valid temperature targets: 9786285

Testing dataset
---------------------------------------------
X shape: (731, 5040, 6)
Y shape: (731, 5040, 2)
Mask shape: (731, 5040, 2)
Date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Input contains NaN: False
Input contains Inf: False
Mask matches target validity: True
Valid rainfal

In [46]:
# Step 35: Daily-to-annual aggregation

years = np.arange(2000, 2026)
num_years = len(years)
num_nodes = rainfall_data.shape[1]

annual_rainfall = np.full(
    (num_years, num_nodes), np.nan, dtype=np.float32
)
annual_temperature = np.full(
    (num_years, num_nodes), np.nan, dtype=np.float32
)

rainfall_valid_days = np.zeros(
    (num_years, num_nodes), dtype=np.int16
)
temperature_valid_days = np.zeros(
    (num_years, num_nodes), dtype=np.int16
)

days_per_year = np.zeros(num_years, dtype=np.int16)

for i, year in enumerate(years):

    year_mask = dates.year == year

    rain_year = rainfall_data[year_mask]
    temp_year = temperature_data[year_mask]

    days_per_year[i] = year_mask.sum()

    rainfall_valid_days[i] = np.isfinite(rain_year).sum(axis=0)
    temperature_valid_days[i] = np.isfinite(temp_year).sum(axis=0)

    # Annual mean from original valid daily observations
    annual_rainfall[i] = np.nanmean(rain_year, axis=0)
    annual_temperature[i] = np.nanmean(temp_year, axis=0)

    print(
        f"{year}: {days_per_year[i]} days | "
        f"Rainfall valid node-years: "
        f"{np.isfinite(annual_rainfall[i]).sum()} | "
        f"Temperature valid node-years: "
        f"{np.isfinite(annual_temperature[i]).sum()}"
    )

# Calculate annual coverage
rainfall_coverage = (
    rainfall_valid_days / days_per_year[:, None]
) * 100

temperature_coverage = (
    temperature_valid_days / days_per_year[:, None]
) * 100

print("\nAnnual aggregation completed.")
print("Rainfall shape:", annual_rainfall.shape)
print("Temperature shape:", annual_temperature.shape)

print("\nOverall annual coverage statistics")
print("Rainfall mean coverage:",
      round(np.nanmean(rainfall_coverage), 2), "%")
print("Temperature mean coverage:",
      round(np.nanmean(temperature_coverage), 2), "%")

print("\nValid-day coverage percentiles")
print("Rainfall:",
      np.nanpercentile(rainfall_coverage, [0, 25, 50, 75, 90, 100]))
print("Temperature:",
      np.nanpercentile(temperature_coverage, [0, 25, 50, 75, 90, 100]))

2000: 366 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4427
2001: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4427
2002: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4427
2003: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4443
2004: 366 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4443


C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\2157181192.py:36: RuntimeWarning: Mean of empty slice
  annual_rainfall[i] = np.nanmean(rain_year, axis=0)
C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\2157181192.py:37: RuntimeWarning: Mean of empty slice
  annual_temperature[i] = np.nanmean(temp_year, axis=0)


2005: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4443
2006: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2007: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2008: 366 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2009: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4443
2010: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4443
2011: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2012: 366 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2013: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2014: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2015: 365 days | Rainfall valid node-years: 4736 | Temperature valid node-years: 4476
2016: 366 days | Rainfall valid node-years: 4736 | Tem

In [47]:
# Step 36: Annual node-year coverage analysis

def summarize_annual_coverage(coverage, variable_name):

    flat = coverage.flatten()

    categories = {
        "No observations (0%)": np.sum(flat == 0),
        "Low coverage (0–50%)": np.sum((flat > 0) & (flat < 50)),
        "Moderate coverage (50–80%)": np.sum((flat >= 50) & (flat < 80)),
        "Good coverage (80–90%)": np.sum((flat >= 80) & (flat < 90)),
        "High coverage (90–99%)": np.sum((flat >= 90) & (flat < 99)),
        "Near-complete coverage (99–<100%)": np.sum((flat >= 99) & (flat < 100)),
        "Complete coverage (100%)": np.sum(flat == 100)
    }

    print(f"\n{variable_name.upper()} — NODE-YEAR COVERAGE")
    print("-" * 55)

    for category, count in categories.items():
        print(f"{category}: {count:,} ({count / len(flat) * 100:.2f}%)")

    print("\nNode-year observations meeting thresholds:")

    for threshold in [80, 90, 95, 99, 100]:
        count = np.sum(flat >= threshold)
        print(
            f"At least {threshold}% coverage: "
            f"{count:,} ({count / len(flat) * 100:.2f}%)"
        )

    return categories


rainfall_coverage_summary = summarize_annual_coverage(
    rainfall_coverage, "Rainfall"
)

temperature_coverage_summary = summarize_annual_coverage(
    temperature_coverage, "Temperature"
)

# Annual counts of valid node-year observations
annual_coverage_table = pd.DataFrame({
    "Year": years,
    "Rainfall_valid_nodes": np.isfinite(annual_rainfall).sum(axis=1),
    "Temperature_valid_nodes": np.isfinite(annual_temperature).sum(axis=1),
    "Rainfall_nodes_90pct": (rainfall_coverage >= 90).sum(axis=1),
    "Temperature_nodes_90pct": (temperature_coverage >= 90).sum(axis=1)
})

print("\nYEAR-WISE COVERAGE SUMMARY")
display(annual_coverage_table)

# Identify nodes with no annual observations in any year
print("\nNodes with no annual observations across all 26 years:")
print(
    "Rainfall:",
    np.sum(~np.isfinite(annual_rainfall).any(axis=0))
)
print(
    "Temperature:",
    np.sum(~np.isfinite(annual_temperature).any(axis=0))
)


RAINFALL — NODE-YEAR COVERAGE
-------------------------------------------------------
No observations (0%): 7,904 (6.03%)
Low coverage (0–50%): 0 (0.00%)
Moderate coverage (50–80%): 0 (0.00%)
Good coverage (80–90%): 0 (0.00%)
High coverage (90–99%): 0 (0.00%)
Near-complete coverage (99–<100%): 68 (0.05%)
Complete coverage (100%): 123,068 (93.92%)

Node-year observations meeting thresholds:
At least 80% coverage: 123,136 (93.97%)
At least 90% coverage: 123,136 (93.97%)
At least 95% coverage: 123,136 (93.97%)
At least 99% coverage: 123,136 (93.97%)
At least 100% coverage: 123,068 (93.92%)

TEMPERATURE — NODE-YEAR COVERAGE
-------------------------------------------------------
No observations (0%): 14,976 (11.43%)
Low coverage (0–50%): 33 (0.03%)
Moderate coverage (50–80%): 98 (0.07%)
Good coverage (80–90%): 507 (0.39%)
High coverage (90–99%): 1,495 (1.14%)
Near-complete coverage (99–<100%): 2,589 (1.98%)
Complete coverage (100%): 111,342 (84.97%)

Node-year observations meeting thresho

,Year,Rainfall_valid_nodes,Temperature_valid_nodes,Rainfall_nodes_90pct,Temperature_nodes_90pct
0,2000,4736,4427,4736,4427
1,2001,4736,4427,4736,4427
2,2002,4736,4427,4736,4394
3,2003,4736,4443,4736,4443
4,2004,4736,4443,4736,4427
5,2005,4736,4443,4736,4427
6,2006,4736,4476,4736,4459
7,2007,4736,4476,4736,4476
8,2008,4736,4476,4736,4427
9,2009,4736,4443,4736,4377



Nodes with no annual observations across all 26 years:
Rainfall: 304
Temperature: 564


In [48]:
# Step 37: Apply 90% annual coverage threshold

ANNUAL_COVERAGE_THRESHOLD = 90.0

# Independent validity masks
rainfall_annual_mask = (
    (rainfall_coverage >= ANNUAL_COVERAGE_THRESHOLD)
    & np.isfinite(annual_rainfall)
)

temperature_annual_mask = (
    (temperature_coverage >= ANNUAL_COVERAGE_THRESHOLD)
    & np.isfinite(annual_temperature)
)

# Preserve annual values only where coverage is sufficient
annual_rainfall_filtered = np.where(
    rainfall_annual_mask,
    annual_rainfall,
    np.nan
).astype(np.float32)

annual_temperature_filtered = np.where(
    temperature_annual_mask,
    annual_temperature,
    np.nan
).astype(np.float32)

print("Annual coverage threshold:", ANNUAL_COVERAGE_THRESHOLD, "%")

print("\nRainfall")
print("Valid node-year observations:", rainfall_annual_mask.sum())
print("Missing node-year observations:", (~rainfall_annual_mask).sum())

print("\nTemperature")
print("Valid node-year observations:", temperature_annual_mask.sum())
print("Missing node-year observations:", (~temperature_annual_mask).sum())

print("\nJoint annual target availability")
joint_annual_mask = rainfall_annual_mask & temperature_annual_mask
print("Both variables valid:", joint_annual_mask.sum())
print("At least one variable valid:",
      (rainfall_annual_mask | temperature_annual_mask).sum())

print("\nAnnual array shapes")
print("Rainfall:", annual_rainfall_filtered.shape)
print("Temperature:", annual_temperature_filtered.shape)

# Verify that the spatial node count remains unchanged
assert annual_rainfall_filtered.shape == (26, 5040)
assert annual_temperature_filtered.shape == (26, 5040)
assert rainfall_annual_mask.shape == (26, 5040)
assert temperature_annual_mask.shape == (26, 5040)

print("\nCoverage filtering completed. All 5,040 nodes retained.")

Annual coverage threshold: 90.0 %

Rainfall
Valid node-year observations: 123136
Missing node-year observations: 7904

Temperature
Valid node-year observations: 115426
Missing node-year observations: 15614

Joint annual target availability
Both variables valid: 112589
At least one variable valid: 125973

Annual array shapes
Rainfall: (26, 5040)
Temperature: (26, 5040)

Coverage filtering completed. All 5,040 nodes retained.


In [49]:
# Step 38: Save annual climate data and quality information

ANNUAL_DIR = PROJECT_DIR / "data" / "intermediate" / "annual_climate"
ANNUAL_DIR.mkdir(parents=True, exist_ok=True)

annual_ds = xr.Dataset(
    data_vars={
        "rainfall": (
            ("year", "node"),
            annual_rainfall_filtered
        ),
        "temperature": (
            ("year", "node"),
            annual_temperature_filtered
        ),
        "rainfall_original_mean": (
            ("year", "node"),
            annual_rainfall.astype(np.float32)
        ),
        "temperature_original_mean": (
            ("year", "node"),
            annual_temperature.astype(np.float32)
        ),
        "rainfall_valid_days": (
            ("year", "node"),
            rainfall_valid_days
        ),
        "temperature_valid_days": (
            ("year", "node"),
            temperature_valid_days
        ),
        "rainfall_coverage_percent": (
            ("year", "node"),
            rainfall_coverage.astype(np.float32)
        ),
        "temperature_coverage_percent": (
            ("year", "node"),
            temperature_coverage.astype(np.float32)
        ),
        "rainfall_valid_mask": (
            ("year", "node"),
            rainfall_annual_mask.astype(np.uint8)
        ),
        "temperature_valid_mask": (
            ("year", "node"),
            temperature_annual_mask.astype(np.uint8)
        )
    },
    coords={
        "year": years,
        "node": np.arange(5040),
        "latitude": ("node", nodes_df["latitude"].values),
        "longitude": ("node", nodes_df["longitude"].values)
    },
    attrs={
        "description": "Annual rainfall and maximum temperature dataset",
        "period": "2000-2025",
        "spatial_nodes": 5040,
        "annual_aggregation": "Mean of original valid daily observations",
        "annual_coverage_threshold_percent": ANNUAL_COVERAGE_THRESHOLD,
        "target_imputation": "Not performed",
        "rainfall_units": "mm/day",
        "temperature_units": "degree_Celsius"
    }
)

annual_ds["rainfall"].attrs["units"] = "mm/day"
annual_ds["temperature"].attrs["units"] = "degree_Celsius"

annual_ds["rainfall_original_mean"].attrs["description"] = (
    "Annual mean before applying the coverage threshold"
)
annual_ds["temperature_original_mean"].attrs["description"] = (
    "Annual mean before applying the coverage threshold"
)

annual_ds["rainfall_valid_mask"].attrs["description"] = (
    "1 indicates annual rainfall coverage of at least 90%"
)
annual_ds["temperature_valid_mask"].attrs["description"] = (
    "1 indicates annual temperature coverage of at least 90%"
)

annual_file = ANNUAL_DIR / "annual_climate_2000_2025.nc"

annual_ds.to_netcdf(
    annual_file,
    engine="netcdf4",
    encoding={
        variable: {"zlib": True, "complevel": 4}
        for variable in annual_ds.data_vars
    }
)

print("Annual dataset saved successfully.")
print("File:", annual_file)
print("\nDataset summary:")
print(annual_ds)
print("\nFile size:",
      round(annual_file.stat().st_size / (1024 ** 2), 2), "MB")

Annual dataset saved successfully.
File: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\intermediate\annual_climate\annual_climate_2000_2025.nc

Dataset summary:
<xarray.Dataset> Size: 4MB
Dimensions:                       (year: 26, node: 5040)
Coordinates:
  * year                          (year) int64 208B 2000 2001 2002 ... 2024 2025
  * node                          (node) int64 40kB 0 1 2 3 ... 5037 5038 5039
    latitude                      (node) float64 40kB 8.451 8.733 ... 29.52
    longitude                     (node) float64 40kB 76.83 76.61 ... 96.03
Data variables:
    rainfall                      (year, node) float32 524kB nan nan ... nan
    temperature                   (year, node) float32 524kB nan 30.93 ... nan
    rainfall_original_mean        (year, node) float32 524kB nan nan ... nan
    temperature_original_mean     (year, node) float32 524kB nan 30.93 ... nan
    rainfall_valid_days     

In [50]:
# Step 39: Verify saved annual dataset

annual_check = xr.open_dataset(annual_file)

print("Reopened dataset:")
print("Dimensions:", dict(annual_check.sizes))
print("Years:", annual_check.year.values[[0, -1]])
print("Nodes:", annual_check.sizes["node"])

# Verify annual values
np.testing.assert_allclose(
    annual_check["rainfall"].values,
    annual_rainfall_filtered,
    rtol=1e-5,
    atol=1e-6,
    equal_nan=True
)

np.testing.assert_allclose(
    annual_check["temperature"].values,
    annual_temperature_filtered,
    rtol=1e-5,
    atol=1e-6,
    equal_nan=True
)

# Verify coverage masks
assert np.array_equal(
    annual_check["rainfall_valid_mask"].values.astype(bool),
    rainfall_annual_mask
)

assert np.array_equal(
    annual_check["temperature_valid_mask"].values.astype(bool),
    temperature_annual_mask
)

# Verify spatial coordinates
np.testing.assert_allclose(
    annual_check.latitude.values,
    nodes_df["latitude"].values
)

np.testing.assert_allclose(
    annual_check.longitude.values,
    nodes_df["longitude"].values
)

# Verify valid observation counts
assert int(annual_check["rainfall_valid_mask"].sum()) == 123136
assert int(annual_check["temperature_valid_mask"].sum()) == 115426

print("\nAll verification checks passed.")
print("Annual values: Verified")
print("Coverage masks: Verified")
print("Spatial coordinates: Verified")
print("Observation counts: Verified")
print("Original 5,040-node structure: Preserved")

Reopened dataset:
Dimensions: {'year': 26, 'node': 5040}
Years: [2000 2025]
Nodes: 5040

All verification checks passed.
Annual values: Verified
Coverage masks: Verified
Spatial coordinates: Verified
Observation counts: Verified
Original 5,040-node structure: Preserved


In [51]:
# Step 40: Define annual forecasting samples

FORECAST_WINDOW = 10

TRAIN_TARGET_YEARS = np.arange(2010, 2018)
VAL_TARGET_YEARS = np.arange(2018, 2024)
TEST_TARGET_YEARS = np.arange(2024, 2026)

year_to_index = {
    int(year): index
    for index, year in enumerate(annual_check.year.values)
}

def create_forecast_indices(target_years):

    input_indices = []
    target_indices = []

    for target_year in target_years:

        target_index = year_to_index[int(target_year)]

        input_start = target_index - FORECAST_WINDOW
        input_end = target_index

        input_indices.append(
            np.arange(input_start, input_end)
        )

        target_indices.append(target_index)

    return np.array(input_indices), np.array(target_indices)


train_input_idx, train_target_idx = create_forecast_indices(
    TRAIN_TARGET_YEARS
)

val_input_idx, val_target_idx = create_forecast_indices(
    VAL_TARGET_YEARS
)

test_input_idx, test_target_idx = create_forecast_indices(
    TEST_TARGET_YEARS
)

# Display forecasting windows
for split_name, target_years, input_indices in [
    ("Training", TRAIN_TARGET_YEARS, train_input_idx),
    ("Validation", VAL_TARGET_YEARS, val_input_idx),
    ("Testing", TEST_TARGET_YEARS, test_input_idx)
]:

    print(f"\n{split_name} samples")

    for target_year, indices in zip(target_years, input_indices):

        input_years = annual_check.year.values[indices]

        print(
            f"Input: {input_years[0]}–{input_years[-1]} "
            f"-> Target: {target_year}"
        )

print("\nSample counts:")
print("Training:", len(train_target_idx))
print("Validation:", len(val_target_idx))
print("Testing:", len(test_target_idx))

assert len(train_target_idx) == 8
assert len(val_target_idx) == 6
assert len(test_target_idx) == 2

assert train_target_idx[-1] < val_target_idx[0]
assert val_target_idx[-1] < test_target_idx[0]

print("\nForecasting sample definitions verified.")


Training samples
Input: 2000–2009 -> Target: 2010
Input: 2001–2010 -> Target: 2011
Input: 2002–2011 -> Target: 2012
Input: 2003–2012 -> Target: 2013
Input: 2004–2013 -> Target: 2014
Input: 2005–2014 -> Target: 2015
Input: 2006–2015 -> Target: 2016
Input: 2007–2016 -> Target: 2017

Validation samples
Input: 2008–2017 -> Target: 2018
Input: 2009–2018 -> Target: 2019
Input: 2010–2019 -> Target: 2020
Input: 2011–2020 -> Target: 2021
Input: 2012–2021 -> Target: 2022
Input: 2013–2022 -> Target: 2023

Testing samples
Input: 2014–2023 -> Target: 2024
Input: 2015–2024 -> Target: 2025

Sample counts:
Training: 8
Validation: 6
Testing: 2

Forecasting sample definitions verified.


In [52]:
# Step 41: Construct annual forecasting arrays

annual_rain = annual_check["rainfall"].values
annual_temp = annual_check["temperature"].values

annual_rain_mask = annual_check["rainfall_valid_mask"].values.astype(bool)
annual_temp_mask = annual_check["temperature_valid_mask"].values.astype(bool)

# Stack rainfall and temperature as two input features
annual_features = np.stack(
    [annual_rain, annual_temp],
    axis=-1
)

# Construct input and target arrays
def build_forecasting_arrays(input_indices, target_indices):

    X = annual_features[input_indices]

    Y = annual_features[target_indices]

    Y_mask = np.stack(
        [
            annual_rain_mask[target_indices],
            annual_temp_mask[target_indices]
        ],
        axis=-1
    )

    return X.astype(np.float32), Y.astype(np.float32), Y_mask


X_train_annual, Y_train_annual, mask_train_annual = (
    build_forecasting_arrays(train_input_idx, train_target_idx)
)

X_val_annual, Y_val_annual, mask_val_annual = (
    build_forecasting_arrays(val_input_idx, val_target_idx)
)

X_test_annual, Y_test_annual, mask_test_annual = (
    build_forecasting_arrays(test_input_idx, test_target_idx)
)

# Display shapes and target availability
for name, X, Y, mask in [
    ("Training", X_train_annual, Y_train_annual, mask_train_annual),
    ("Validation", X_val_annual, Y_val_annual, mask_val_annual),
    ("Testing", X_test_annual, Y_test_annual, mask_test_annual)
]:

    print(f"\n{name}")
    print("Input shape:", X.shape)
    print("Target shape:", Y.shape)
    print("Target mask shape:", mask.shape)
    print("Valid rainfall targets:", mask[..., 0].sum())
    print("Valid temperature targets:", mask[..., 1].sum())
    print("Finite input values:", np.isfinite(X).sum())

# Integrity checks
assert X_train_annual.shape == (8, 10, 5040, 2)
assert X_val_annual.shape == (6, 10, 5040, 2)
assert X_test_annual.shape == (2, 10, 5040, 2)

assert Y_train_annual.shape == (8, 5040, 2)
assert Y_val_annual.shape == (6, 5040, 2)
assert Y_test_annual.shape == (2, 5040, 2)

assert np.array_equal(mask_train_annual, np.isfinite(Y_train_annual))
assert np.array_equal(mask_val_annual, np.isfinite(Y_val_annual))
assert np.array_equal(mask_test_annual, np.isfinite(Y_test_annual))

print("\nAnnual forecasting arrays constructed successfully.")


Training
Input shape: (8, 10, 5040, 2)
Target shape: (8, 5040, 2)
Target mask shape: (8, 5040, 2)
Valid rainfall targets: 37888
Valid temperature targets: 35563
Finite input values: 733285

Validation
Input shape: (6, 10, 5040, 2)
Target shape: (6, 5040, 2)
Target mask shape: (6, 5040, 2)
Valid rainfall targets: 28416
Valid temperature targets: 26627
Finite input values: 550446

Testing
Input shape: (2, 10, 5040, 2)
Target shape: (2, 5040, 2)
Target mask shape: (2, 5040, 2)
Valid rainfall targets: 9472
Valid temperature targets: 8952
Finite input values: 183750

Annual forecasting arrays constructed successfully.


In [53]:
# Step 42: Annual input missingness analysis

def analyse_input_missingness(name, X, input_indices):

    print(f"\n{name.upper()} INPUT MISSINGNESS")
    print("-" * 55)

    for feature_index, feature_name in enumerate(
        ["Rainfall", "Temperature"]
    ):

        feature_data = X[..., feature_index]
        total = feature_data.size
        missing = np.isnan(feature_data).sum()

        print(f"\n{feature_name}")
        print("Total input values:", f"{total:,}")
        print("Missing values:", f"{missing:,}")
        print("Missing percentage:", round(missing / total * 100, 2), "%")

        # Missing values for each forecasting sample
        for sample_index, indices in enumerate(input_indices):

            sample = feature_data[sample_index]
            missing_count = np.isnan(sample).sum()

            print(
                f"  Target {years[indices[-1] + 1]}: "
                f"{missing_count:,} missing "
                f"({missing_count / sample.size * 100:.2f}%)"
            )


analyse_input_missingness(
    "Training", X_train_annual, train_input_idx
)

analyse_input_missingness(
    "Validation", X_val_annual, val_input_idx
)

analyse_input_missingness(
    "Testing", X_test_annual, test_input_idx
)


TRAINING INPUT MISSINGNESS
-------------------------------------------------------

Rainfall
Total input values: 403,200
Missing values: 24,320
Missing percentage: 6.03 %
  Target 2010: 3,040 missing (6.03%)
  Target 2011: 3,040 missing (6.03%)
  Target 2012: 3,040 missing (6.03%)
  Target 2013: 3,040 missing (6.03%)
  Target 2014: 3,040 missing (6.03%)
  Target 2015: 3,040 missing (6.03%)
  Target 2016: 3,040 missing (6.03%)
  Target 2017: 3,040 missing (6.03%)

Temperature
Total input values: 403,200
Missing values: 48,795
Missing percentage: 12.1 %
  Target 2010: 6,116 missing (12.13%)
  Target 2011: 6,149 missing (12.20%)
  Target 2012: 6,149 missing (12.20%)
  Target 2013: 6,083 missing (12.07%)
  Target 2014: 6,132 missing (12.17%)
  Target 2015: 6,083 missing (12.07%)
  Target 2016: 6,050 missing (12.00%)
  Target 2017: 6,033 missing (11.97%)

VALIDATION INPUT MISSINGNESS
-------------------------------------------------------

Rainfall
Total input values: 302,400
Missing value

In [54]:
# Step 43: Calculate training-only node-wise imputation statistics

# Training input data only: 2000–2016
rain_train_inputs = X_train_annual[..., 0]
temp_train_inputs = X_train_annual[..., 1]

# Node-wise mean across training samples and 10-year windows
rain_node_mean = np.nanmean(
    rain_train_inputs,
    axis=(0, 1)
)

temp_node_mean = np.nanmean(
    temp_train_inputs,
    axis=(0, 1)
)

# Global training means
rain_global_mean = np.nanmean(rain_train_inputs)
temp_global_mean = np.nanmean(temp_train_inputs)

# Replace undefined node means with global training means
rain_node_mean = np.where(
    np.isfinite(rain_node_mean),
    rain_node_mean,
    rain_global_mean
)

temp_node_mean = np.where(
    np.isfinite(temp_node_mean),
    temp_node_mean,
    temp_global_mean
)

print("Training-only imputation statistics")
print("\nRainfall")
print("Global mean:", round(float(rain_global_mean), 4))
print("Node means:", np.isfinite(rain_node_mean).sum())
print("Nodes using global fallback:",
      np.sum(~np.isfinite(np.nanmean(rain_train_inputs, axis=(0, 1)))))

print("\nTemperature")
print("Global mean:", round(float(temp_global_mean), 4))
print("Node means:", np.isfinite(temp_node_mean).sum())
print("Nodes using global fallback:",
      np.sum(~np.isfinite(np.nanmean(temp_train_inputs, axis=(0, 1)))))

Training-only imputation statistics

Rainfall
Global mean: 3.0487
Node means: 5040
Nodes using global fallback: 304

Temperature
Global mean: 31.2854
Node means: 5040
Nodes using global fallback: 564


C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\1196274262.py:8: RuntimeWarning: Mean of empty slice
  rain_node_mean = np.nanmean(
C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\1196274262.py:13: RuntimeWarning: Mean of empty slice
  temp_node_mean = np.nanmean(
C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\1196274262.py:40: RuntimeWarning: Mean of empty slice
  np.sum(~np.isfinite(np.nanmean(rain_train_inputs, axis=(0, 1)))))
C:\Users\rajgm\AppData\Local\Temp\ipykernel_3808\1196274262.py:46: RuntimeWarning: Mean of empty slice
  np.sum(~np.isfinite(np.nanmean(temp_train_inputs, axis=(0, 1)))))


In [55]:
# Step 44: Impute annual inputs and add missingness indicators

def impute_annual_inputs(X, rain_means, temp_means):

    X_imputed = np.array(X, dtype=np.float32, copy=True)

    # Original missingness indicators
    rain_missing = ~np.isfinite(X_imputed[..., 0])
    temp_missing = ~np.isfinite(X_imputed[..., 1])

    # Replace missing values using training-only node means
    rain_replacement = np.broadcast_to(
        rain_means[None, None, :],
        X_imputed[..., 0].shape
    )

    temp_replacement = np.broadcast_to(
        temp_means[None, None, :],
        X_imputed[..., 1].shape
    )

    X_imputed[..., 0] = np.where(
        rain_missing,
        rain_replacement,
        X_imputed[..., 0]
    )

    X_imputed[..., 1] = np.where(
        temp_missing,
        temp_replacement,
        X_imputed[..., 1]
    )

    # Add missingness indicators as two additional features
    X_final = np.concatenate(
        [
            X_imputed,
            rain_missing[..., None].astype(np.float32),
            temp_missing[..., None].astype(np.float32)
        ],
        axis=-1
    )

    return X_final


X_train_annual_imp = impute_annual_inputs(
    X_train_annual, rain_node_mean, temp_node_mean
)

X_val_annual_imp = impute_annual_inputs(
    X_val_annual, rain_node_mean, temp_node_mean
)

X_test_annual_imp = impute_annual_inputs(
    X_test_annual, rain_node_mean, temp_node_mean
)

for name, X in [
    ("Training", X_train_annual_imp),
    ("Validation", X_val_annual_imp),
    ("Testing", X_test_annual_imp)
]:

    print(f"\n{name}")
    print("Shape:", X.shape)
    print("Finite values:", np.isfinite(X).sum())
    print("Remaining missing values:", np.isnan(X).sum())
    print("Rainfall missing indicators:", int(X[..., 2].sum()))
    print("Temperature missing indicators:", int(X[..., 3].sum()))

    assert np.isfinite(X).all()
    assert X.shape[-1] == 4

print("\nAnnual input imputation completed.")


Training
Shape: (8, 10, 5040, 4)
Finite values: 1612800
Remaining missing values: 0
Rainfall missing indicators: 24320
Temperature missing indicators: 48795

Validation
Shape: (6, 10, 5040, 4)
Finite values: 1209600
Remaining missing values: 0
Rainfall missing indicators: 18240
Temperature missing indicators: 36114

Testing
Shape: (2, 10, 5040, 4)
Finite values: 403200
Remaining missing values: 0
Rainfall missing indicators: 6080
Temperature missing indicators: 11770

Annual input imputation completed.


In [56]:
# Step 45: Calculate training-only normalization statistics

# Calculate statistics from the imputed training inputs
rain_mean = float(X_train_annual_imp[..., 0].mean())
rain_std = float(X_train_annual_imp[..., 0].std())

temp_mean = float(X_train_annual_imp[..., 1].mean())
temp_std = float(X_train_annual_imp[..., 1].std())

print("Training-only normalization statistics")

print("\nRainfall")
print("Mean:", round(rain_mean, 6))
print("Standard deviation:", round(rain_std, 6))

print("\nMaximum temperature")
print("Mean:", round(temp_mean, 6))
print("Standard deviation:", round(temp_std, 6))

assert rain_std > 0
assert temp_std > 0

print("\nNormalization statistics calculated successfully.")

Training-only normalization statistics

Rainfall
Mean: 3.048718
Standard deviation: 2.088418

Maximum temperature
Mean: 31.23583
Standard deviation: 2.896394

Normalization statistics calculated successfully.


In [57]:
# Step 46: Standardize climate features using training statistics

def normalize_annual_inputs(X):

    X_norm = X.copy()

    # Normalize rainfall
    X_norm[..., 0] = (
        X[..., 0] - rain_mean
    ) / rain_std

    # Normalize maximum temperature
    X_norm[..., 1] = (
        X[..., 1] - temp_mean
    ) / temp_std

    # Keep missingness indicators unchanged
    return X_norm.astype(np.float32)


X_train_final = normalize_annual_inputs(X_train_annual_imp)
X_val_final = normalize_annual_inputs(X_val_annual_imp)
X_test_final = normalize_annual_inputs(X_test_annual_imp)

for name, X in [
    ("Training", X_train_final),
    ("Validation", X_val_final),
    ("Testing", X_test_final)
]:

    print(f"\n{name}")
    print("Shape:", X.shape)
    print("All values finite:", np.isfinite(X).all())

    print("Rainfall normalized mean:",
          round(float(X[..., 0].mean()), 4))
    print("Rainfall normalized std:",
          round(float(X[..., 0].std()), 4))

    print("Temperature normalized mean:",
          round(float(X[..., 1].mean()), 4))
    print("Temperature normalized std:",
          round(float(X[..., 1].std()), 4))

    print("Rainfall indicator values:",
          np.unique(X[..., 2]))
    print("Temperature indicator values:",
          np.unique(X[..., 3]))

assert np.isfinite(X_train_final).all()
assert np.isfinite(X_val_final).all()
assert np.isfinite(X_test_final).all()

print("\nAnnual input normalization completed.")


Training
Shape: (8, 10, 5040, 4)
All values finite: True
Rainfall normalized mean: 0.0
Rainfall normalized std: 1.0
Temperature normalized mean: 0.0
Temperature normalized std: 1.0
Rainfall indicator values: [0. 1.]
Temperature indicator values: [0. 1.]

Validation
Shape: (6, 10, 5040, 4)
All values finite: True
Rainfall normalized mean: 0.008
Rainfall normalized std: 1.0117
Temperature normalized mean: 0.0104
Temperature normalized std: 1.0037
Rainfall indicator values: [0. 1.]
Temperature indicator values: [0. 1.]

Testing
Shape: (2, 10, 5040, 4)
All values finite: True
Rainfall normalized mean: 0.0342
Rainfall normalized std: 1.0393
Temperature normalized mean: -0.0012
Temperature normalized std: 0.9967
Rainfall indicator values: [0. 1.]
Temperature indicator values: [0. 1.]

Annual input normalization completed.


In [58]:
# Step 47: Final integrity checks for annual forecasting data

# Verify targets are unchanged
assert np.array_equal(
    mask_train_annual, np.isfinite(Y_train_annual)
)
assert np.array_equal(
    mask_val_annual, np.isfinite(Y_val_annual)
)
assert np.array_equal(
    mask_test_annual, np.isfinite(Y_test_annual)
)

# Verify input arrays are finite
assert np.isfinite(X_train_final).all()
assert np.isfinite(X_val_final).all()
assert np.isfinite(X_test_final).all()

# Verify feature dimensions
assert X_train_final.shape == (8, 10, 5040, 4)
assert X_val_final.shape == (6, 10, 5040, 4)
assert X_test_final.shape == (2, 10, 5040, 4)

# Verify target dimensions
assert Y_train_annual.shape == (8, 5040, 2)
assert Y_val_annual.shape == (6, 5040, 2)
assert Y_test_annual.shape == (2, 5040, 2)

# Verify missingness indicators remain binary
for X in [X_train_final, X_val_final, X_test_final]:
    assert set(np.unique(X[..., 2])).issubset({0.0, 1.0})
    assert set(np.unique(X[..., 3])).issubset({0.0, 1.0})

print("FINAL INTEGRITY CHECKS")
print("----------------------")
print("Training input:", X_train_final.shape)
print("Validation input:", X_val_final.shape)
print("Testing input:", X_test_final.shape)

print("\nTraining valid targets:")
print("Rainfall:", mask_train_annual[..., 0].sum())
print("Temperature:", mask_train_annual[..., 1].sum())

print("\nValidation valid targets:")
print("Rainfall:", mask_val_annual[..., 0].sum())
print("Temperature:", mask_val_annual[..., 1].sum())

print("\nTesting valid targets:")
print("Rainfall:", mask_test_annual[..., 0].sum())
print("Temperature:", mask_test_annual[..., 1].sum())

print("\nAll integrity checks passed.")

FINAL INTEGRITY CHECKS
----------------------
Training input: (8, 10, 5040, 4)
Validation input: (6, 10, 5040, 4)
Testing input: (2, 10, 5040, 4)

Training valid targets:
Rainfall: 37888
Temperature: 35563

Validation valid targets:
Rainfall: 28416
Temperature: 26627

Testing valid targets:
Rainfall: 9472
Temperature: 8952

All integrity checks passed.


In [59]:
# Step 48: Save final annual forecasting dataset

ANNUAL_MODEL_DIR = (
    PROJECT_DIR / "data" / "processed" / "annual_model_ready"
)
ANNUAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save training data
np.savez_compressed(
    ANNUAL_MODEL_DIR / "training_annual.npz",
    X=X_train_final,
    Y=Y_train_annual,
    Y_mask=mask_train_annual,
    input_indices=train_input_idx,
    target_indices=train_target_idx,
    target_years=TRAIN_TARGET_YEARS
)

# Save validation data
np.savez_compressed(
    ANNUAL_MODEL_DIR / "validation_annual.npz",
    X=X_val_final,
    Y=Y_val_annual,
    Y_mask=mask_val_annual,
    input_indices=val_input_idx,
    target_indices=val_target_idx,
    target_years=VAL_TARGET_YEARS
)

# Save testing data
np.savez_compressed(
    ANNUAL_MODEL_DIR / "testing_annual.npz",
    X=X_test_final,
    Y=Y_test_annual,
    Y_mask=mask_test_annual,
    input_indices=test_input_idx,
    target_indices=test_target_idx,
    target_years=TEST_TARGET_YEARS
)

# Save preprocessing metadata
import json

metadata = {
    "forecasting_task": "Annual one-year-ahead forecasting",
    "input_window_years": FORECAST_WINDOW,
    "historical_period": "2000-2025",
    "training_target_years": TRAIN_TARGET_YEARS.tolist(),
    "validation_target_years": VAL_TARGET_YEARS.tolist(),
    "testing_target_years": TEST_TARGET_YEARS.tolist(),
    "number_of_nodes": 5040,
    "number_of_input_features": 4,
    "input_features": [
        "Normalized rainfall",
        "Normalized maximum temperature",
        "Rainfall missingness indicator",
        "Temperature missingness indicator"
    ],
    "target_variables": [
        "Rainfall (mm/day)",
        "Maximum temperature (degree_Celsius)"
    ],
    "annual_coverage_threshold_percent": ANNUAL_COVERAGE_THRESHOLD,
    "imputation_method": "Training-only node-wise mean; global training mean fallback",
    "normalization": {
        "rainfall_mean": rain_mean,
        "rainfall_std": rain_std,
        "temperature_mean": temp_mean,
        "temperature_std": temp_std
    },
    "target_imputation": False,
    "graph": {
        "file": "data/spatial/static_spatial_graph_verified.pt",
        "number_of_nodes": 5040,
        "number_of_directed_edges": 40320,
        "construction": "Haversine k-nearest neighbours",
        "k": 8
    }
}

metadata_file = ANNUAL_MODEL_DIR / "annual_forecasting_metadata.json"

with open(metadata_file, "w") as f:
    json.dump(metadata, f, indent=4)

print("Final annual forecasting datasets saved.")

for file in sorted(ANNUAL_MODEL_DIR.iterdir()):
    if file.is_file():
        print(
            file.name,
            "|",
            round(file.stat().st_size / (1024 ** 2), 2),
            "MB"
        )

print("\nMetadata saved:", metadata_file)

Final annual forecasting datasets saved.
annual_forecasting_metadata.json | 0.0 MB
testing_annual.npz | 0.8 MB
training_annual.npz | 3.18 MB
validation_annual.npz | 2.39 MB

Metadata saved: D:\SVKM NMIMS Indore\Academic Related\Research Papers\Spatiotemporal GNN\New Code Setup\Spatiotemporal_GNN_Revised\data\processed\annual_model_ready\annual_forecasting_metadata.json


In [60]:
# Step 49: Independent verification of saved annual datasets

import json
import numpy as np

# Load metadata
with open(metadata_file, "r") as f:
    saved_metadata = json.load(f)

# Verify each split
saved_splits = {
    "Training": (
        "training_annual.npz",
        X_train_final,
        Y_train_annual,
        mask_train_annual
    ),
    "Validation": (
        "validation_annual.npz",
        X_val_final,
        Y_val_annual,
        mask_val_annual
    ),
    "Testing": (
        "testing_annual.npz",
        X_test_final,
        Y_test_annual,
        mask_test_annual
    )
}

for name, (filename, X_original, Y_original, mask_original) in saved_splits.items():

    with np.load(ANNUAL_MODEL_DIR / filename, allow_pickle=False) as saved:

        X_saved = saved["X"]
        Y_saved = saved["Y"]
        mask_saved = saved["Y_mask"]

        assert np.array_equal(X_saved, X_original)
        assert np.array_equal(Y_saved, Y_original, equal_nan=True)
        assert np.array_equal(mask_saved, mask_original)

        print(f"\n{name}: VERIFIED")
        print("Input shape:", X_saved.shape)
        print("Target shape:", Y_saved.shape)
        print("Valid rainfall targets:", mask_saved[..., 0].sum())
        print("Valid temperature targets:", mask_saved[..., 1].sum())

# Verify metadata
assert saved_metadata["number_of_nodes"] == 5040
assert saved_metadata["number_of_input_features"] == 4
assert saved_metadata["input_window_years"] == 10
assert saved_metadata["target_imputation"] is False

print("\nMetadata verification: PASSED")
print("All saved annual forecasting datasets verified successfully.")


Training: VERIFIED
Input shape: (8, 10, 5040, 4)
Target shape: (8, 5040, 2)
Valid rainfall targets: 37888
Valid temperature targets: 35563

Validation: VERIFIED
Input shape: (6, 10, 5040, 4)
Target shape: (6, 5040, 2)
Valid rainfall targets: 28416
Valid temperature targets: 26627

Testing: VERIFIED
Input shape: (2, 10, 5040, 4)
Target shape: (2, 5040, 2)
Valid rainfall targets: 9472
Valid temperature targets: 8952

Metadata verification: PASSED
All saved annual forecasting datasets verified successfully.
